# Agentic Payment System

An LLM-powered multi-agent payment platform that combines autonomous AI agents with deterministic business logic to simulate secure digital payment workflows.

## Overview

The Agentic Payment System explores how specialized LLM agents can collaborate within a real-world financial workflow while keeping critical payment operations deterministic and controlled.

The system interprets natural-language requests, coordinates payment operations, evaluates transactions for potential fraud, performs security reviews, reflects on failed operations, critiques system responses, and generates user-friendly explanations.

Rather than allowing an LLM to directly execute financial operations, the architecture separates **AI reasoning from business execution**. LLM agents handle routing, reasoning, evaluation, and explanations, while deterministic Python tools enforce payment rules and execute operations.

This creates a more reliable architecture for integrating LLM-based decision-making with systems that require predictable business logic.

> **Note:** This project is an educational simulation and does not connect to real bank accounts, credit cards, or payment services.

## Key Features

- Natural-language payment requests
- LLM-based intent detection and routing
- Multi-agent workflow orchestration
- Automated fraud detection
- Security review of payment operations
- Contextual business memory
- Structured LLM outputs
- Reflection and critique mechanisms
- User-friendly explanations
- Fallback handling for unsupported requests
- Deterministic payment execution and validation

## Multi-Agent Architecture

The system uses specialized agents with clearly separated responsibilities:

- **Router Agent** – Identifies user intent and determines the appropriate workflow
- **Orchestrator Agent** – Coordinates agents and payment operations
- **Fraud Detection Agent** – Evaluates transactions for suspicious activity
- **Security Agent** – Performs additional security analysis
- **Reflection Agent** – Analyzes failed operations and system behavior
- **Critic Agent** – Evaluates the quality and consistency of generated results
- **Explanation Agent** – Converts system decisions into understandable explanations
- **Fallback Agent** – Handles unsupported or ambiguous requests

## Architecture Philosophy

A central design principle of the project is the separation between **LLM reasoning** and **deterministic execution**.

LLM agents are responsible for:

- Intent recognition
- Workflow coordination
- Fraud and security evaluation
- Reflection and critique
- Natural-language explanations

Deterministic Python components are responsible for:

- User and wallet operations
- Balance validation
- Money transfers
- Payment requests
- Transaction management
- Business-rule enforcement

This separation helps maintain predictable payment behavior while still taking advantage of LLM reasoning capabilities.

## LLM Integration

The system integrates the OpenAI API using structured outputs and Pydantic models to provide predictable communication between the language model and application logic.

Contextual business memory is maintained across operations, allowing agents to reason about recent users, transactions, payment requests, and system actions.

## Technologies

- Python
- OpenAI API
- LLM Agents
- Pydantic
- Structured Outputs
- Multi-Agent Systems
- Prompt Engineering
- Business Logic Validation

In [27]:
%%capture
!pip install -q ipython-autotime
%load_ext autotime

time: 6.92 s (started: 2026-09-28 12:26:35 +00:00)


In [28]:
!pip install -q openai
!pip install -q pydantic
!pip install -q python-dotenv
!pip install -q gradio

time: 35.9 s (started: 2026-09-28 12:26:42 +00:00)


time: 35.9 s (started: 2026-09-28 12:26:42 +00:00)


### Global Test Configuration

Controls whether test blocks are executed during notebook development.
Set to `True` to enable tests and `False` to skip them.

In [29]:
RUN_COMPONENT_TESTS = True

time: 526 µs (started: 2026-09-28 12:27:18 +00:00)


## Libraries and Dependencies

The system is organized into two core layers:

1. **LLM Agent Layer** – Communicates with the OpenAI API and powers the specialized AI agents.
2. **Deterministic Python Layer** – Manages payment operations, business memory, logging, and business-rule enforcement.

**Pydantic** is used to define structured outputs and validate responses returned by the language model, ensuring reliable communication between the LLM agents and the application logic.

In [30]:
import os
import json
import gradio as gr

import pandas as pd

from abc import ABC, abstractmethod
from dataclasses import dataclass, asdict
from datetime import datetime
from enum import Enum
from typing import Any, Dict, List, Optional, Type, TypeVar
from pprint import pprint

from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI

time: 1.54 ms (started: 2026-09-28 12:27:18 +00:00)


## API Key Configuration

The OpenAI API key is loaded securely from **Google Colab Secrets** when running in Colab, with an environment variable used as a fallback for other environments.

The API key is never hard-coded in the notebook, allowing the project to be shared publicly without exposing credentials.

In [31]:
try:
    from google.colab import userdata

    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

except ImportError:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


def validate_api_key() -> None:
    if not OPENAI_API_KEY:
        raise ValueError(
            "Missing OPENAI_API_KEY. "
            "Add it to Google Colab Secrets or set it as an environment variable."
        )

    print("OpenAI API key loaded successfully.")
validate_api_key()

OpenAI API key loaded successfully.
time: 613 ms (started: 2026-09-28 12:27:18 +00:00)


## Language Model Configuration

The system uses OpenAI as its LLM provider. All specialized agents operate through the same model provider while maintaining independent responsibilities, prompts, and structured output schemas.

This design preserves clear separation between agent roles while providing a consistent interface for model interactions.

In [32]:
OPENAI_MODEL = "gpt-5-mini"

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)

print(
    "OpenAI client was created successfully."
)

print(
    "Model:",
    OPENAI_MODEL
)

OpenAI client was created successfully.
Model: gpt-5-mini
time: 39 ms (started: 2026-09-28 12:27:19 +00:00)


## Standardized Agent Result

All agents return a standardized `AgentResult` object, providing a consistent interface for communication across the multi-agent system.

Each result contains:

- **Agent name** – Identifies the agent that produced the result.
- **Output** – Contains the structured response generated by the agent.
- **Confidence score** – Represents the agent's confidence in its result.
- **Metadata** – Stores additional information such as the model provider, model name, and execution status.

Using a shared result structure allows the orchestrator to process outputs from different agents consistently, even when each agent has a different role and system prompt.

In [33]:
@dataclass
class AgentResult:
    agent_name: str
    output: Any
    confidence: float = 1.0
    metadata: Optional[Dict[str, Any]] = None

    def display(self) -> None:
        print("=" * 60)
        print("Agent name:", self.agent_name)
        print("Confidence:", self.confidence)
        print("Output:")
        print(json.dumps(
            self.output,
            indent=2,
            ensure_ascii=False,
            default=str
        ))
        print("Metadata:")
        print(json.dumps(
            self.metadata or {},
            indent=2,
            ensure_ascii=False,
            default=str
        ))
        print("=" * 60)

time: 2.55 ms (started: 2026-09-28 12:27:19 +00:00)


### Basic Validation

In [34]:
example_result = AgentResult(
    agent_name="ExampleLLMAgent",
    output={
        "status": "success",
        "message": "The shared result structure is working."
    },
    confidence=0.95,
    metadata={
        "provider": "example",
        "model": "example-model"
    }
)

example_result.display()

Agent name: ExampleLLMAgent
Confidence: 0.95
Output:
{
  "status": "success",
  "message": "The shared result structure is working."
}
Metadata:
{
  "provider": "example",
  "model": "example-model"
}
time: 1.52 ms (started: 2026-09-28 12:27:19 +00:00)


## LLM Provider Abstraction

To keep the agent architecture independent of a specific language-model provider, the system defines a common `LLMProvider` interface.

Each provider implements a `generate_json()` method that receives:

- A **system prompt** defining the agent's role and behavior.
- A **user prompt** containing the current task or request.
- A **Pydantic model** defining the required structured output.

This abstraction makes the architecture modular and allows the underlying LLM implementation to be replaced or extended without modifying the business logic of individual agents.

In [35]:
StructuredOutput = TypeVar(
    "StructuredOutput",
    bound=BaseModel
)


class LLMProvider(ABC):
    provider_name: str
    model_name: str

    @abstractmethod
    def generate_json(
        self,
        system_prompt: str,
        user_prompt: str,
        output_model: Type[StructuredOutput]
    ) -> StructuredOutput:
        raise NotImplementedError

time: 995 µs (started: 2026-09-28 12:27:19 +00:00)


### OpenAI Provider
The `OpenAIProvider` implements the common LLM provider interface using the OpenAI API.

It sends the agent's system and user prompts to the configured model and uses Pydantic schemas to parse the response into a validated structured object. Invalid or empty responses are handled explicitly to prevent malformed LLM output from propagating through the system.

In [36]:
class OpenAIProvider(LLMProvider):
    def __init__(
        self,
        client: OpenAI,
        model_name: str = OPENAI_MODEL
    ):
        self.client = client
        self.provider_name = "OpenAI"
        self.model_name = model_name

    def generate_json(
        self,
        system_prompt: str,
        user_prompt: str,
        output_model: Type[StructuredOutput]
    ) -> StructuredOutput:

        try:
            response = self.client.responses.parse(
                model=self.model_name,
                input=[
                    {
                        "role": "system",
                        "content": system_prompt
                    },
                    {
                        "role": "user",
                        "content": user_prompt
                    }
                ],
                text_format=output_model
            )

            parsed_result = response.output_parsed

            if parsed_result is None:
                raise RuntimeError(
                    "OpenAI returned an empty structured response."
                )

            return parsed_result

        except ValidationError as error:
            raise RuntimeError(
                "OpenAI returned an invalid structured response."
            ) from error

        except Exception as error:
            raise RuntimeError(
                f"OpenAI API request failed: {error}"
            ) from error

time: 2.37 ms (started: 2026-09-28 12:27:19 +00:00)


### Provider Initialization

In [37]:
openai_provider = OpenAIProvider(
    client=openai_client
)

print(
    "OpenAI provider:",
    openai_provider.model_name
)

OpenAI provider: gpt-5-mini
time: 2.15 ms (started: 2026-09-28 12:27:19 +00:00)


### Provider Configuration Validation

Before using the provider within the agent system, its configuration is validated to ensure that the expected provider and model have been initialized correctly.

In [38]:
assert (
    openai_provider.provider_name
    == "OpenAI"
)

assert (
    openai_provider.model_name
    == OPENAI_MODEL
)

print(
    "OpenAI provider was configured successfully."
)

OpenAI provider was configured successfully.
time: 937 µs (started: 2026-09-28 12:27:19 +00:00)


## Router Agent

The `RouterAgent` acts as the entry point to the multi-agent workflow. It analyzes natural-language user requests and converts them into structured business intents that can be processed by the rest of the system.

Its responsibilities include:

- Identifying the requested payment operation.
- Extracting explicitly provided business parameters.
- Returning a structured intent classification.
- Providing a confidence score for the classification.

The router does not execute financial operations itself. Its responsibility is limited to understanding and structuring the request, while execution is delegated to the deterministic payment layer.


In [39]:
class Intent(str, Enum):

    CREATE_USER = "createUser"

    CHECK_BALANCE = "checkBalance"

    TRANSFER_MONEY = "transferMoney"

    REQUEST_PAYMENT = "requestPayment"

    APPROVE_PAYMENT = "approvePayment"

    REJECT_PAYMENT = "rejectPayment"

    SHOW_TRANSACTIONS = "showTransactions"

    FRAUD_CHECK = "fraudCheck"

    SECURITY_REVIEW = "securityReview"

    EXPLAIN_LAST_ACTION = "explainLastAction"

    UNKNOWN = "unknown"

time: 1.28 ms (started: 2026-09-28 12:27:19 +00:00)


### Router Parameters

The router extracts only the parameters explicitly provided in the user's request.

These parameters represent the information required by different payment operations, including user identifiers, transaction amounts, names, and payment-request IDs.

Missing information is intentionally left undefined rather than inferred by the LLM, helping prevent the system from introducing unsupported values into financial operations.

In [40]:
class RouterParameters(BaseModel):
    """
    Structured parameters extracted from the user request.
    """

    name: Optional[str] = Field(
        default=None,
        description="The name of a new user."
    )

    phone_number: Optional[str] = Field(
        default=None,
        description="The phone number of a new user."
    )

    initial_balance: Optional[float] = Field(
        default=None,
        description="The initial balance of a new user."
    )

    user_id: Optional[int] = Field(
        default=None,
        description=(
            "The unique numeric user identifier used for "
            "balance checks and transaction history."
        )
    )

    sender_id: Optional[int] = Field(
        default=None,
        description="The unique numeric identifier of the money sender."
    )

    receiver_id: Optional[int] = Field(
        default=None,
        description="The unique numeric identifier of the money receiver."
    )

    requester_id: Optional[int] = Field(
        default=None,
        description=(
            "The unique numeric identifier of the user "
            "requesting payment."
        )
    )

    payer_id: Optional[int] = Field(
        default=None,
        description=(
            "The unique numeric identifier of the user "
            "expected to pay."
        )
    )

    amount: Optional[float] = Field(
        default=None,
        description="The monetary amount involved in the request."
    )

    request_id: Optional[int] = Field(
        default=None,
        description="The unique numeric identifier of a payment request."
    )



class RouterOutput(BaseModel):
    """
    Structured output returned by RouterAgent.
    """

    intent: Intent = Field(
        description="The detected business intent."
    )

    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Confidence in the detected intent."
    )

    explanation: str = Field(
        description="A short explanation of the classification."
    )

    parameters: RouterParameters = Field(
        default_factory=RouterParameters,
        description="Business parameters extracted from the request."
    )

time: 8.88 ms (started: 2026-09-28 12:27:19 +00:00)


## RouterAgent Instructions

The `RouterAgent` is guided by a dedicated system prompt that defines its responsibilities, supported intents, and expected output structure.

Instead of relying on hard-coded conditional logic or regular expressions to interpret user requests, the agent uses the LLM to analyze natural-language input, identify the appropriate intent, extract relevant parameters, and return a structured response.

This approach allows the router to handle variations in user phrasing while maintaining a consistent interface for downstream components.

In [41]:
ROUTER_SYSTEM_PROMPT = """
You are the RouterAgent of an educational digital payment system.

Your only responsibilities are:

1. Classify the user's request.
2. Extract business parameters that explicitly appear in the request.
3. Return a confidence score and a short explanation.

Choose exactly one of the following intents:

- createUser
- checkBalance
- transferMoney
- requestPayment
- approvePayment
- rejectPayment
- showTransactions
- fraudCheck
- securityReview
- explainLastAction
- unknown

The available parameter fields are:

- name
- phone_number
- initial_balance
- user_id
- sender_id
- receiver_id
- requester_id
- payer_id
- amount
- request_id

Parameter rules:

- For createUser:
  - use name;
  - use phone_number;
  - use initial_balance.

- For checkBalance:
  - use user_id.

- For showTransactions:
  - use user_id.

- For transferMoney:
  - use sender_id;
  - use receiver_id;
  - use amount.

- For requestPayment:
  - use requester_id;
  - use payer_id;
  - use amount.

- For approvePayment:
  - use request_id.

- For rejectPayment:
  - use request_id.

Important identification rules:

- User names are not unique identifiers.
- Financial operations must use numeric user identifiers.
- Do not place a person's name in an identifier field.
- Do not convert a name into an identifier.
- Do not invent missing identifiers.
- Do not infer identifiers from names.
- A name may be extracted only into the name field for createUser.

Missing-data rules:

- A request may still have a valid intent even when required parameters are missing.
- When a required parameter is missing, leave it as null.
- Do not change the intent to unknown only because a parameter is missing.
- Use unknown only when the requested action itself cannot be identified.

Keep the explanation short and factual.
"""

time: 1.01 ms (started: 2026-09-28 12:27:19 +00:00)


### Router Agent Implementation

The `RouterAgent` transforms natural-language requests into structured, actionable intents.

It sends the user request to the LLM together with the router's system prompt and validates the response using the predefined Pydantic schema. Only explicitly provided parameters are extracted, reducing the risk of introducing unsupported information.

The validated response is then converted into the standardized `AgentResult` format, allowing it to be processed consistently by downstream components.

In [42]:
class RouterAgent:
    """
    LLM-based agent responsible for intent classification
    and parameter extraction.
    """

    def __init__(
        self,
        provider: LLMProvider
    ):
        self.provider = provider

    def run(
        self,
        user_message: str
    ) -> AgentResult:
        """
        Analyze a user request and return a structured route.
        """

        if not user_message or not user_message.strip():
            raise ValueError(
                "The user message cannot be empty."
            )

        result = self.provider.generate_json(
            system_prompt=ROUTER_SYSTEM_PROMPT,
            user_prompt=user_message,
            output_model=RouterOutput
        )

        return AgentResult(
            agent_name="RouterAgent",
            output=result.model_dump(
                exclude_none=True
            ),
            confidence=result.confidence,
            metadata={
                "provider": self.provider.provider_name,
                "model": self.provider.model_name,
                "status": "success"
            }
        )

time: 1.23 ms (started: 2026-09-28 12:27:19 +00:00)


## RouterAgent Initialization

The `RouterAgent` is initialized with the configured `openai_provider`.

The agent uses the OpenAI model to identify the appropriate business intent and extract the relevant parameters from natural-language requests.

Passing the LLM provider through the constructor keeps the routing logic independent from the underlying model implementation, making it easier to replace or extend the provider without modifying the agent's core logic.


In [43]:
router_agent = RouterAgent(
    openai_provider
)

time: 546 µs (started: 2026-09-28 12:27:19 +00:00)


<div dir="rtl">


### Tests and validation

#### User Creation Test

This test verifies that the `RouterAgent` correctly identifies a user-creation request and extracts the user's name, phone number, and initial balance.

The user ID is not provided as part of the request. It is generated later by the `PaymentSystem` during user creation.

In [45]:
if RUN_COMPONENT_TESTS:
  create_user_route_test = router_agent.run(
      "Create a new user named Alice "
      "with phone 0501234567 and balance 800"
  )

  create_user_route_test.display()

Agent name: RouterAgent
Confidence: 0.95
Output:
{
  "intent": "createUser",
  "confidence": 0.95,
  "explanation": "User creation request with name, phone number, and initial balance provided.",
  "parameters": {
    "name": "Alice",
    "phone_number": "0501234567",
    "initial_balance": 800.0
  }
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "success"
}
time: 4.7 s (started: 2026-09-28 12:29:00 +00:00)


#### User Creation Routing Validation

This test verifies that the intent and parameters returned by the `RouterAgent` correctly match the user's request.

In [46]:
if RUN_COMPONENT_TESTS:
  assert (
      create_user_route_test.agent_name
      == "RouterAgent"
  )

  assert (
      create_user_route_test.output["intent"]
      == Intent.CREATE_USER.value
  )

  assert (
      create_user_route_test.output["parameters"]["name"]
      == "Alice"
  )

  assert (
      create_user_route_test.output["parameters"]["phone_number"]
      == "0501234567"
  )

  assert (
      create_user_route_test.output["parameters"]["initial_balance"]
      == 800.0
  )

  print("RouterAgent create-user test passed.")

RouterAgent create-user test passed.
time: 908 µs (started: 2026-09-28 12:29:09 +00:00)


#### Money Transfer Routing Test

Financial operations reference users through numeric identifiers.

This test verifies that the `RouterAgent` correctly extracts the sender ID, receiver ID, and transfer amount from the user's request.


In [47]:
if RUN_COMPONENT_TESTS:
  transfer_route_test = router_agent.run(
      "Transfer 250 from user 1 to user 2"
  )

  transfer_route_test.display()

Agent name: RouterAgent
Confidence: 0.98
Output:
{
  "intent": "transferMoney",
  "confidence": 0.98,
  "explanation": "User asked to transfer 250 from user 1 to user 2.",
  "parameters": {
    "sender_id": 1,
    "receiver_id": 2,
    "amount": 250.0
  }
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "success"
}
time: 5.57 s (started: 2026-09-28 12:29:13 +00:00)


#### Money Transfer Routing Validation

This test verifies that the detected intent is `transferMoney` and that all required numeric parameters are extracted correctly.

In [48]:
if RUN_COMPONENT_TESTS:
  assert (
      transfer_route_test.output["intent"]
      == Intent.TRANSFER_MONEY.value
  )

  assert (
      transfer_route_test.output["parameters"]["sender_id"]
      == 1
  )

  assert (
      transfer_route_test.output["parameters"]["receiver_id"]
      == 2
  )

  assert (
      transfer_route_test.output["parameters"]["amount"]
      == 250.0
  )

  print("RouterAgent transfer test passed.")

RouterAgent transfer test passed.
time: 857 µs (started: 2026-09-28 12:29:23 +00:00)


#### Transfer Request Without User IDs

This test evaluates how the `RouterAgent` handles a transfer request that contains user names but does not provide numeric user IDs.

The agent may identify the `transferMoney` intent and extract the transfer amount, but it must not infer or fabricate sender or receiver IDs.

The `OrchestratorAgent` will later detect the missing required parameters and route the request to the appropriate fallback mechanism.

In [49]:
if RUN_COMPONENT_TESTS:
  missing_ids_route_test = router_agent.run(
      "Transfer 150 from Alice to Bob"
  )

  missing_ids_route_test.display()

Agent name: RouterAgent
Confidence: 0.87
Output:
{
  "intent": "transferMoney",
  "confidence": 0.87,
  "explanation": "User asked to transfer 150 but provided names (Alice, Bob) instead of numeric user IDs; amount parsed.",
  "parameters": {
    "amount": 150.0
  }
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "success"
}
time: 6.11 s (started: 2026-09-28 12:29:26 +00:00)


#### Identifier Hallucination Prevention Validation

This test verifies that the `RouterAgent` does not convert user names into numeric identifiers or fabricate information that was not explicitly provided in the request.

In [50]:
if RUN_COMPONENT_TESTS:
  assert (
      missing_ids_route_test.output["intent"]
      == Intent.TRANSFER_MONEY.value
  )

  assert (
      missing_ids_route_test.output["parameters"]["amount"]
      == 150.0
  )

  assert (
      "sender_id"
      not in missing_ids_route_test.output["parameters"]
  )

  assert (
      "receiver_id"
      not in missing_ids_route_test.output["parameters"]
  )

  print("RouterAgent missing-identifiers test passed.")

RouterAgent missing-identifiers test passed.
time: 905 µs (started: 2026-09-28 12:29:35 +00:00)


# Deterministic Payment System

The `PaymentSystem` is the deterministic Python layer responsible for managing application data and executing financial operations.

LLM agents do not directly modify balances or create transactions. Instead, they determine which action should be performed, while the `PaymentSystem` validates and executes the operation according to explicit business rules.

The system is built around four core entities:

1. **User** – Represents a registered user of the system.
2. **Wallet** – Stores and manages a user's balance.
3. **Transaction** – Represents a money transfer between users.
4. **PaymentRequest** – Represents a payment request between users.

In [51]:
@dataclass
class User:
    user_id: int
    name: str
    phone_number: str


@dataclass
class Wallet:
    user_id: int
    balance: float


@dataclass
class Transaction:
    transaction_id: int
    sender_id: int
    receiver_id: int
    amount: float
    timestamp: datetime
    status: str
    risk_score: float = 0.0


@dataclass
class PaymentRequest:
    request_id: int
    requester_id: int
    payer_id: int
    amount: float
    status: str
    created_at: datetime

time: 8.69 ms (started: 2026-09-28 12:29:38 +00:00)


### Business Entity Validation

This test creates sample instances of each core business entity to verify that their data structures are initialized and stored correctly.

The validation is isolated from the `PaymentSystem` and does not modify application state or execute any financial operation.

In [52]:
example_user = User(
    user_id=1,
    name="Alice",
    phone_number="0501234567"
)

example_wallet = Wallet(
    user_id=1,
    balance=800.0
)

example_transaction = Transaction(
    transaction_id=1,
    sender_id=1,
    receiver_id=2,
    amount=100.0,
    timestamp=datetime.now(),
    status="completed"
)

example_payment_request = PaymentRequest(
    request_id=1,
    requester_id=2,
    payer_id=1,
    amount=50.0,
    status="pending",
    created_at=datetime.now()
)

pprint(example_user)
pprint(example_wallet)
pprint(example_transaction)
pprint(example_payment_request)

User(user_id=1, name='Alice', phone_number='0501234567')
Wallet(user_id=1, balance=800.0)
Transaction(transaction_id=1,
            sender_id=1,
            receiver_id=2,
            amount=100.0,
            timestamp=datetime.datetime(2026, 9, 28, 12, 29, 50, 471220),
            status='completed',
            risk_score=0.0)
PaymentRequest(request_id=1,
               requester_id=2,
               payer_id=1,
               amount=50.0,
               status='pending',
               created_at=datetime.datetime(2026, 9, 28, 12, 29, 50, 471310))
time: 2.32 ms (started: 2026-09-28 12:29:50 +00:00)


## PaymentSystem Class

The `PaymentSystem` serves as the deterministic core of the payment platform and manages the system's application state.

It maintains:

- **Users** – Registered users in the system.
- **Wallets** – User balances and wallet information.
- **Transactions** – Completed money transfers between users.
- **Payment Requests** – Payment requests and their current status.
- **Audit Log** – Records of significant system operations.

Internal counters are used to generate unique identifiers for users, transactions, and payment requests.

Significant operations are recorded in the `audit_log`, providing traceability across the payment workflow.

In [53]:
class PaymentSystem:
    def __init__(self):
        self.users: Dict[int, User] = {}
        self.wallets: Dict[int, Wallet] = {}
        self.transactions: List[Transaction] = []
        self.payment_requests: Dict[int, PaymentRequest] = {}
        self.audit_log: List[Dict[str, Any]] = []
        self.log_audit = self.audit_log

        self.next_user_id = 1
        self.next_transaction_id = 1
        self.next_request_id = 1

    def _add_audit_log(
        self,
        action: str,
        status: str,
        details: Optional[Dict[str, Any]] = None
    ) -> None:
        self.audit_log.append({
            "timestamp": datetime.now(),
            "action": action,
            "status": status,
            "details": details or {}
        })

    def _success_result(
        self,
        action: str,
        output: Dict[str, Any],
        metadata: Optional[Dict[str, Any]] = None
    ) -> AgentResult:
        result_metadata = {
            "action": action,
            "status": "success"
        }

        if metadata:
            result_metadata.update(metadata)

        return AgentResult(
            agent_name="PaymentSystem",
            output=output,
            confidence=1.0,
            metadata=result_metadata
        )

    def _error_result(
        self,
        action: str,
        message: str,
        output: Optional[Dict[str, Any]] = None,
        metadata: Optional[Dict[str, Any]] = None
    ) -> AgentResult:
        result_output = {
            "success": False,
            "message": message
        }

        if output:
            result_output.update(output)

        result_metadata = {
            "action": action,
            "status": "rejected"
        }

        if metadata:
            result_metadata.update(metadata)

        return AgentResult(
            agent_name="PaymentSystem",
            output=result_output,
            confidence=1.0,
            metadata=result_metadata
        )

    def create_user(
        self,
        name: str,
        phone_number: str,
        initial_balance: float
    ) -> AgentResult:
        action = "create_user"

        if initial_balance < 0:
            message = (
                "A user cannot be created with a negative "
                "initial balance."
            )

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "name": name,
                    "phone_number": phone_number,
                    "initial_balance": initial_balance,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message
            )

        user_id = self.next_user_id

        user = User(
            user_id=user_id,
            name=name,
            phone_number=phone_number
        )

        wallet = Wallet(
            user_id=user_id,
            balance=float(initial_balance)
        )

        self.users[user_id] = user
        self.wallets[user_id] = wallet
        self.next_user_id += 1

        self._add_audit_log(
            action=action,
            status="success",
            details={
                "user_id": user_id,
                "name": name,
                "phone_number": phone_number,
                "initial_balance": initial_balance
            }
        )

        return self._success_result(
            action=action,
            output={
                "success": True,
                "message": "User created successfully.",
                "user": asdict(user),
                "wallet": asdict(wallet)
            },
            metadata={
                "user_id": user_id
            }
        )

    def get_balance(
        self,
        user_id: int
    ) -> AgentResult:
        action = "get_balance"

        if user_id not in self.users:
            message = "User does not exist."

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "user_id": user_id,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message,
                metadata={
                    "user_id": user_id
                }
            )

        wallet = self.wallets[user_id]

        self._add_audit_log(
            action=action,
            status="success",
            details={
                "user_id": user_id,
                "balance": wallet.balance
            }
        )

        return self._success_result(
            action=action,
            output={
                "success": True,
                "user_id": user_id,
                "balance": wallet.balance
            },
            metadata={
                "user_id": user_id
            }
        )

    def transfer_money(
        self,
        sender_id: int,
        receiver_id: int,
        amount: float
    ) -> AgentResult:
        action = "transfer_money"

        if amount <= 0:
            message = "Transfer amount must be greater than zero."

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "sender_id": sender_id,
                    "receiver_id": receiver_id,
                    "amount": amount,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message
            )

        if sender_id not in self.users:
            message = "Sender does not exist."

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "sender_id": sender_id,
                    "receiver_id": receiver_id,
                    "amount": amount,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message
            )

        if receiver_id not in self.users:
            message = "Receiver does not exist."

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "sender_id": sender_id,
                    "receiver_id": receiver_id,
                    "amount": amount,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message
            )

        if sender_id == receiver_id:
            message = "A user cannot transfer money to themselves."

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "sender_id": sender_id,
                    "receiver_id": receiver_id,
                    "amount": amount,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message
            )

        sender_wallet = self.wallets[sender_id]
        receiver_wallet = self.wallets[receiver_id]

        if sender_wallet.balance < amount:
            message = "Insufficient balance."

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "sender_id": sender_id,
                    "receiver_id": receiver_id,
                    "amount": amount,
                    "sender_balance": sender_wallet.balance,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message
            )

        sender_balance_before = sender_wallet.balance
        receiver_balance_before = receiver_wallet.balance

        sender_wallet.balance -= amount
        receiver_wallet.balance += amount

        transaction = Transaction(
            transaction_id=self.next_transaction_id,
            sender_id=sender_id,
            receiver_id=receiver_id,
            amount=float(amount),
            timestamp=datetime.now(),
            status="completed",
            risk_score=0.0
        )

        self.transactions.append(transaction)
        self.next_transaction_id += 1

        self._add_audit_log(
            action=action,
            status="success",
            details={
                "transaction_id": transaction.transaction_id,
                "sender_id": sender_id,
                "receiver_id": receiver_id,
                "amount": amount,
                "sender_balance_before": sender_balance_before,
                "sender_balance_after": sender_wallet.balance,
                "receiver_balance_before": receiver_balance_before,
                "receiver_balance_after": receiver_wallet.balance
            }
        )

        return self._success_result(
            action=action,
            output={
                "success": True,
                "message": "Transfer completed successfully.",
                "transaction": asdict(transaction),
                "sender_balance": sender_wallet.balance,
                "receiver_balance": receiver_wallet.balance
            },
            metadata={
                "transaction_id": transaction.transaction_id,
                "sender_balance_before": sender_balance_before
            }
        )

    def get_transactions(
        self,
        user_id: int
    ) -> AgentResult:
        action = "get_transactions"

        if user_id not in self.users:
            message = "User does not exist."

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "user_id": user_id,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message,
                output={
                    "transactions": []
                },
                metadata={
                    "user_id": user_id
                }
            )

        user_transactions = [
            asdict(transaction)
            for transaction in self.transactions
            if (
                transaction.sender_id == user_id
                or transaction.receiver_id == user_id
            )
        ]

        transaction_count = len(user_transactions)

        self._add_audit_log(
            action=action,
            status="success",
            details={
                "user_id": user_id,
                "transaction_count": transaction_count
            }
        )

        return self._success_result(
            action=action,
            output={
                "success": True,
                "user_id": user_id,
                "transactions": user_transactions
            },
            metadata={
                "user_id": user_id,
                "transaction_count": transaction_count
            }
        )

    def request_payment(
        self,
        requester_id: int,
        payer_id: int,
        amount: float
    ) -> AgentResult:
        action = "request_payment"

        if amount <= 0:
            message = (
                "Payment request amount must be greater than zero."
            )

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "requester_id": requester_id,
                    "payer_id": payer_id,
                    "amount": amount,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message
            )

        if requester_id not in self.users:
            message = "Requester does not exist."

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "requester_id": requester_id,
                    "payer_id": payer_id,
                    "amount": amount,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message
            )

        if payer_id not in self.users:
            message = "Payer does not exist."

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "requester_id": requester_id,
                    "payer_id": payer_id,
                    "amount": amount,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message
            )

        payment_request = PaymentRequest(
            request_id=self.next_request_id,
            requester_id=requester_id,
            payer_id=payer_id,
            amount=float(amount),
            status="pending",
            created_at=datetime.now()
        )

        self.payment_requests[
            payment_request.request_id
        ] = payment_request

        self.next_request_id += 1

        self._add_audit_log(
            action=action,
            status="success",
            details={
                "request_id": payment_request.request_id,
                "requester_id": requester_id,
                "payer_id": payer_id,
                "amount": amount
            }
        )

        return self._success_result(
            action=action,
            output={
                "success": True,
                "message": "Payment request created successfully.",
                "payment_request": asdict(payment_request)
            },
            metadata={
                "request_id": payment_request.request_id
            }
        )

    def approve_payment_request(
        self,
        request_id: int
    ) -> AgentResult:
        action = "approve_payment_request"

        if request_id not in self.payment_requests:
            message = "Payment request does not exist."

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "request_id": request_id,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message,
                metadata={
                    "request_id": request_id
                }
            )

        payment_request = self.payment_requests[request_id]

        if payment_request.status != "pending":
            message = (
                "Payment request has already been approved or rejected."
            )

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "request_id": request_id,
                    "current_status": payment_request.status,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message,
                output={
                    "payment_request": asdict(payment_request)
                },
                metadata={
                    "request_id": request_id
                }
            )

        transfer_result = self.transfer_money(
            sender_id=payment_request.payer_id,
            receiver_id=payment_request.requester_id,
            amount=payment_request.amount
        )

        if transfer_result.output["success"] is False:
            message = (
                "Payment request could not be approved: "
                + transfer_result.output["message"]
            )

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "request_id": request_id,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message,
                output={
                    "payment_request": asdict(payment_request)
                },
                metadata={
                    "request_id": request_id
                }
            )

        payment_request.status = "approved"
        transaction = transfer_result.output["transaction"]

        self._add_audit_log(
            action=action,
            status="success",
            details={
                "request_id": request_id,
                "transaction_id": transaction["transaction_id"]
            }
        )

        return self._success_result(
            action=action,
            output={
                "success": True,
                "message": "Payment request approved successfully.",
                "payment_request": asdict(payment_request),
                "transaction": transaction
            },
            metadata={
                "request_id": request_id,
                "transaction_id": transaction["transaction_id"]
            }
        )

    def reject_payment_request(
        self,
        request_id: int
    ) -> AgentResult:
        action = "reject_payment_request"

        if request_id not in self.payment_requests:
            message = "Payment request does not exist."

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "request_id": request_id,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message,
                metadata={
                    "request_id": request_id
                }
            )

        payment_request = self.payment_requests[request_id]

        if payment_request.status != "pending":
            message = (
                "Payment request has already been approved or rejected."
            )

            self._add_audit_log(
                action=action,
                status="rejected",
                details={
                    "request_id": request_id,
                    "current_status": payment_request.status,
                    "reason": message
                }
            )

            return self._error_result(
                action=action,
                message=message,
                output={
                    "payment_request": asdict(payment_request)
                },
                metadata={
                    "request_id": request_id
                }
            )

        payment_request.status = "rejected"

        self._add_audit_log(
            action=action,
            status="success",
            details={
                "request_id": request_id
            }
        )

        return self._success_result(
            action=action,
            output={
                "success": True,
                "message": "Payment request rejected successfully.",
                "payment_request": asdict(payment_request)
            },
            metadata={
                "request_id": request_id
            }
        )

time: 24.7 ms (started: 2026-09-28 12:29:53 +00:00)


### User Creation Validation

This test creates two users with valid initial balances and verifies that they are successfully added to the system.

It then attempts to create a user with a negative initial balance to confirm that the operation is rejected and that the system state remains unchanged.

In [55]:
if RUN_COMPONENT_TESTS:
  payment_system = PaymentSystem()

  create_user_test_1 = payment_system.create_user(
      name="Alice",
      phone_number="0501234567",
      initial_balance=800.0
  )

  create_user_test_2 = payment_system.create_user(
      name="Bob",
      phone_number="0507654321",
      initial_balance=500.0
  )

  negative_balance_test = payment_system.create_user(
      name="Invalid User",
      phone_number="0500000000",
      initial_balance=-100.0
  )

  create_user_test_1.display()
  create_user_test_2.display()
  negative_balance_test.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "message": "User created successfully.",
  "user": {
    "user_id": 1,
    "name": "Alice",
    "phone_number": "0501234567"
  },
  "wallet": {
    "user_id": 1,
    "balance": 800.0
  }
}
Metadata:
{
  "action": "create_user",
  "status": "success",
  "user_id": 1
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "message": "User created successfully.",
  "user": {
    "user_id": 2,
    "name": "Bob",
    "phone_number": "0507654321"
  },
  "wallet": {
    "user_id": 2,
    "balance": 500.0
  }
}
Metadata:
{
  "action": "create_user",
  "status": "success",
  "user_id": 2
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": false,
  "message": "A user cannot be created with a negative initial balance."
}
Metadata:
{
  "action": "create_user",
  "status": "rejected"
}
time: 1.33 ms (started: 2026-09-28 12:34:03 +00:00)


### User Creation Validation Results

The validation confirms that:

- Two valid users are successfully created.
- Each user receives a unique identifier.
- A corresponding wallet is created for each user.
- Negative initial balances are rejected.
- Rejected users are not added to the system.

In [56]:
if RUN_COMPONENT_TESTS:
  assert create_user_test_1.output["success"] is True
  assert create_user_test_2.output["success"] is True

  assert create_user_test_1.output["user"]["user_id"] == 1
  assert create_user_test_2.output["user"]["user_id"] == 2

  assert payment_system.wallets[1].balance == 800.0
  assert payment_system.wallets[2].balance == 500.0

  assert negative_balance_test.output["success"] is False

  assert len(payment_system.users) == 2
  assert len(payment_system.wallets) == 2

  assert payment_system.next_user_id == 3

  print("PaymentSystem create-user tests passed.")

PaymentSystem create-user tests passed.
time: 1.61 ms (started: 2026-09-28 12:34:09 +00:00)


### Audit Log Validation

Significant system operations are recorded in the `audit_log` to provide traceability and support monitoring of payment-system activity.

This test verifies that the audit log contains records for two successful operations and one rejected operation.

In [57]:
if RUN_COMPONENT_TESTS:
  for log_entry in payment_system.audit_log:
      print(json.dumps(
          log_entry,
          indent=2,
          ensure_ascii=False,
          default=str
      ))

{
  "timestamp": "2026-09-28 12:34:03.875477",
  "action": "create_user",
  "status": "success",
  "details": {
    "user_id": 1,
    "name": "Alice",
    "phone_number": "0501234567",
    "initial_balance": 800.0
  }
}
{
  "timestamp": "2026-09-28 12:34:03.875528",
  "action": "create_user",
  "status": "success",
  "details": {
    "user_id": 2,
    "name": "Bob",
    "phone_number": "0507654321",
    "initial_balance": 500.0
  }
}
{
  "timestamp": "2026-09-28 12:34:03.875541",
  "action": "create_user",
  "status": "rejected",
  "details": {
    "name": "Invalid User",
    "phone_number": "0500000000",
    "initial_balance": -100.0,
    "reason": "A user cannot be created with a negative initial balance."
  }
}
time: 2.04 ms (started: 2026-09-28 12:34:14 +00:00)


### Audit Log Verification

This test verifies that all user-creation attempts are recorded in the `audit_log`, including the rejected operation.

In [58]:
if RUN_COMPONENT_TESTS:
  assert len(payment_system.audit_log) == 3

  assert payment_system.audit_log[0]["status"] == "success"
  assert payment_system.audit_log[1]["status"] == "success"
  assert payment_system.audit_log[2]["status"] == "rejected"

  assert (
      payment_system.audit_log[2]["action"]
      == "create_user"
  )

  print("Audit-log validation passed.")

Audit-log validation passed.
time: 1.25 ms (started: 2026-09-28 12:34:17 +00:00)


### Balance Validation

This test verifies the balances of both users after their creation.

It also attempts to retrieve the balance of a non-existent user to confirm that the system handles invalid user IDs correctly.

In [59]:
if RUN_COMPONENT_TESTS:
  balance_test_1 = payment_system.get_balance(
      user_id=1
  )

  balance_test_2 = payment_system.get_balance(
      user_id=2
  )

  missing_user_balance_test = payment_system.get_balance(
      user_id=999
  )

  balance_test_1.display()
  balance_test_2.display()
  missing_user_balance_test.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "user_id": 1,
  "balance": 800.0
}
Metadata:
{
  "action": "get_balance",
  "status": "success",
  "user_id": 1
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "user_id": 2,
  "balance": 500.0
}
Metadata:
{
  "action": "get_balance",
  "status": "success",
  "user_id": 2
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": false,
  "message": "User does not exist."
}
Metadata:
{
  "action": "get_balance",
  "status": "rejected",
  "user_id": 999
}
time: 1.05 ms (started: 2026-09-28 12:34:19 +00:00)


### Balance Validation Results

The validation confirms that the stored balances match the users' initial balances and that balance requests for non-existent users are rejected.

In [60]:
if RUN_COMPONENT_TESTS:
  assert balance_test_1.output["success"] is True
  assert balance_test_1.output["balance"] == 800.0

  assert balance_test_2.output["success"] is True
  assert balance_test_2.output["balance"] == 500.0

  assert missing_user_balance_test.output["success"] is False
  assert (
      missing_user_balance_test.output["message"]
      == "User does not exist."
  )

  print("PaymentSystem balance tests passed.")

PaymentSystem balance tests passed.
time: 843 µs (started: 2026-09-28 12:34:23 +00:00)


### Valid Money Transfer Test

This test performs a transfer of 200 from User 1 to User 2.

Before the transfer:

- User 1 has a balance of 800.
- User 2 has a balance of 500.

After the transfer, the expected balances are 600 for User 1 and 700 for User 2.

In [61]:
if RUN_COMPONENT_TESTS:
  successful_transfer_test = payment_system.transfer_money(
      sender_id=1,
      receiver_id=2,
      amount=200.0
  )

  successful_transfer_test.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "message": "Transfer completed successfully.",
  "transaction": {
    "transaction_id": 1,
    "sender_id": 1,
    "receiver_id": 2,
    "amount": 200.0,
    "timestamp": "2026-09-28 12:34:26.921812",
    "status": "completed",
    "risk_score": 0.0
  },
  "sender_balance": 600.0,
  "receiver_balance": 700.0
}
Metadata:
{
  "action": "transfer_money",
  "status": "success",
  "transaction_id": 1,
  "sender_balance_before": 800.0
}
time: 1.83 ms (started: 2026-09-28 12:34:26 +00:00)


### Valid Money Transfer Validation

The validation confirms that:

- The transfer is completed successfully.
- Both wallet balances are updated correctly.
- A new transaction is created.
- The transaction is stored in the transaction history.

In [62]:
if RUN_COMPONENT_TESTS:
  assert successful_transfer_test.output["success"] is True

  assert payment_system.wallets[1].balance == 600.0
  assert payment_system.wallets[2].balance == 700.0

  assert len(payment_system.transactions) == 1

  assert (
      payment_system.transactions[0].transaction_id
      == 1
  )

  assert (
      successful_transfer_test.output["transaction"]["amount"]
      == 200.0
  )

  print("PaymentSystem valid-transfer test passed.")

PaymentSystem valid-transfer test passed.
time: 1.08 ms (started: 2026-09-28 12:34:30 +00:00)


### Money Transfer Edge Case Tests

The payment system enforces several business rules to prevent invalid transfers.

The following edge cases are tested to ensure that the system rejects:

- Negative transfer amounts.
- Zero-value transfers.
- Transfers with insufficient funds.
- Transfers to a non-existent user.
- Transfers from a non-existent user.
- Transfers where the sender and receiver are the same user.

In [63]:
if RUN_COMPONENT_TESTS:
  negative_transfer_test = payment_system.transfer_money(
      sender_id=1,
      receiver_id=2,
      amount=-50.0
  )

  zero_transfer_test = payment_system.transfer_money(
      sender_id=1,
      receiver_id=2,
      amount=0.0
  )

  insufficient_balance_test = payment_system.transfer_money(
      sender_id=1,
      receiver_id=2,
      amount=5000.0
  )

  missing_receiver_test = payment_system.transfer_money(
      sender_id=1,
      receiver_id=999,
      amount=50.0
  )

  missing_sender_test = payment_system.transfer_money(
      sender_id=999,
      receiver_id=2,
      amount=50.0
  )

  self_transfer_test = payment_system.transfer_money(
      sender_id=1,
      receiver_id=1,
      amount=50.0
  )

  negative_transfer_test.display()
  zero_transfer_test.display()
  insufficient_balance_test.display()
  missing_receiver_test.display()
  missing_sender_test.display()
  self_transfer_test.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": false,
  "message": "Transfer amount must be greater than zero."
}
Metadata:
{
  "action": "transfer_money",
  "status": "rejected"
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": false,
  "message": "Transfer amount must be greater than zero."
}
Metadata:
{
  "action": "transfer_money",
  "status": "rejected"
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": false,
  "message": "Insufficient balance."
}
Metadata:
{
  "action": "transfer_money",
  "status": "rejected"
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": false,
  "message": "Receiver does not exist."
}
Metadata:
{
  "action": "transfer_money",
  "status": "rejected"
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": false,
  "message": "Sender does not exist."
}
Metadata:
{
  "action": "transfer_money",
  "status": "rejected"
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success":

### Money Transfer Edge Case Validation

The validation confirms that all invalid transfer attempts are rejected and that wallet balances remain unchanged after the failed operations.

In [64]:
if RUN_COMPONENT_TESTS:
  assert negative_transfer_test.output["success"] is False
  assert zero_transfer_test.output["success"] is False
  assert insufficient_balance_test.output["success"] is False
  assert missing_receiver_test.output["success"] is False
  assert missing_sender_test.output["success"] is False
  assert self_transfer_test.output["success"] is False

  assert payment_system.wallets[1].balance == 600.0
  assert payment_system.wallets[2].balance == 700.0

  assert len(payment_system.transactions) == 1

  print("PaymentSystem transfer edge-case tests passed.")

PaymentSystem transfer edge-case tests passed.
time: 1.52 ms (started: 2026-09-28 12:34:42 +00:00)


### Transaction History Test

This test creates a fresh payment system with two users and executes a single money transfer between them.

It then verifies that the same transaction appears in the transaction history of both the sender and the receiver.

In [65]:
if RUN_COMPONENT_TESTS:
  transaction_history_system = PaymentSystem()

  transaction_history_system.create_user(
      name="Alice",
      phone_number="0501234567",
      initial_balance=800.0
  )

  transaction_history_system.create_user(
      name="Bob",
      phone_number="0507654321",
      initial_balance=500.0
  )

  transaction_history_system.transfer_money(
      sender_id=1,
      receiver_id=2,
      amount=200.0
  )

  user_1_transactions = (
      transaction_history_system.get_transactions(
          user_id=1
      )
  )

  user_2_transactions = (
      transaction_history_system.get_transactions(
          user_id=2
      )
  )

  missing_user_transactions = (
      transaction_history_system.get_transactions(
          user_id=999
      )
  )

  user_1_transactions.display()
  user_2_transactions.display()
  missing_user_transactions.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "user_id": 1,
  "transactions": [
    {
      "transaction_id": 1,
      "sender_id": 1,
      "receiver_id": 2,
      "amount": 200.0,
      "timestamp": "2026-09-28 12:34:45.254766",
      "status": "completed",
      "risk_score": 0.0
    }
  ]
}
Metadata:
{
  "action": "get_transactions",
  "status": "success",
  "user_id": 1,
  "transaction_count": 1
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "user_id": 2,
  "transactions": [
    {
      "transaction_id": 1,
      "sender_id": 1,
      "receiver_id": 2,
      "amount": 200.0,
      "timestamp": "2026-09-28 12:34:45.254766",
      "status": "completed",
      "risk_score": 0.0
    }
  ]
}
Metadata:
{
  "action": "get_transactions",
  "status": "success",
  "user_id": 2,
  "transaction_count": 1
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": false,
  "message": "User does not exist.",
  "transactions": []
}
Me

### Transaction History Validation

The validation confirms that the transaction is correctly stored and accessible in the transaction history of both the sender and the receiver.

It also verifies that transaction-history requests for non-existent users are rejected.


In [66]:
if RUN_COMPONENT_TESTS:
  assert user_1_transactions.output["success"] is True
  assert user_2_transactions.output["success"] is True

  assert (
      len(user_1_transactions.output["transactions"])
      == 1
  )

  assert (
      len(user_2_transactions.output["transactions"])
      == 1
  )

  assert (
      user_1_transactions.output[
          "transactions"
      ][0]["transaction_id"]
      == 1
  )

  assert (
      user_2_transactions.output[
          "transactions"
      ][0]["transaction_id"]
      == 1
  )

  assert missing_user_transactions.output["success"] is False
  assert missing_user_transactions.output["transactions"] == []

  print("PaymentSystem transaction-history tests passed.")

PaymentSystem transaction-history tests passed.
time: 2.57 ms (started: 2026-09-28 12:34:50 +00:00)


### Payment Request Creation and Approval Test

In this test, User 1 creates a payment request for User 2.

The request is initially created with a `pending` status.

When User 2 approves the request, the system transfers the requested amount from User 2 to User 1 and updates the payment request status to `approved`.

In [67]:
if RUN_COMPONENT_TESTS:
  payment_request_system = PaymentSystem()

  payment_request_system.create_user(
      name="Alice",
      phone_number="0501234567",
      initial_balance=800.0
  )

  payment_request_system.create_user(
      name="Bob",
      phone_number="0507654321",
      initial_balance=500.0
  )

  payment_request_test = (
      payment_request_system.request_payment(
          requester_id=1,
          payer_id=2,
          amount=100.0
      )
  )

  approve_request_test = (
      payment_request_system.approve_payment_request(
          request_id=1
      )
  )

  payment_request_test.display()
  approve_request_test.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "message": "Payment request created successfully.",
  "payment_request": {
    "request_id": 1,
    "requester_id": 1,
    "payer_id": 2,
    "amount": 100.0,
    "status": "pending",
    "created_at": "2026-09-28 12:35:03.456267"
  }
}
Metadata:
{
  "action": "request_payment",
  "status": "success",
  "request_id": 1
}
Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "message": "Payment request approved successfully.",
  "payment_request": {
    "request_id": 1,
    "requester_id": 1,
    "payer_id": 2,
    "amount": 100.0,
    "status": "approved",
    "created_at": "2026-09-28 12:35:03.456267"
  },
  "transaction": {
    "transaction_id": 1,
    "sender_id": 2,
    "receiver_id": 1,
    "amount": 100.0,
    "timestamp": "2026-09-28 12:35:03.456344",
    "status": "completed",
    "risk_score": 0.0
  }
}
Metadata:
{
  "action": "approve_payment_request",
  "status": "success",
  "request

### Payment Request Approval Validation

The validation confirms that:

- The payment request is created successfully.
- The request status transitions from `pending` to `approved`.
- A corresponding transaction is created.
- The wallet balances are updated correctly.

In [68]:
if RUN_COMPONENT_TESTS:
  assert payment_request_test.output["success"] is True

  assert (
      payment_request_test.output[
          "payment_request"
      ]["status"]
      == "pending"
  )

  assert approve_request_test.output["success"] is True

  assert (
      payment_request_system.payment_requests[1].status
      == "approved"
  )

  assert (
      payment_request_system.wallets[1].balance
      == 900.0
  )

  assert (
      payment_request_system.wallets[2].balance
      == 400.0
  )

  assert len(payment_request_system.transactions) == 1

  print("Payment-request approval tests passed.")

Payment-request approval tests passed.
time: 1.17 ms (started: 2026-09-28 12:35:07 +00:00)


### Duplicate Payment Approval Test

According to the system's business rules, a payment request that has already been approved or rejected cannot be approved again.

This test attempts to approve the payment request from the previous test a second time to verify that duplicate approval is rejected.

In [69]:
if RUN_COMPONENT_TESTS:
  double_approval_test = (
      payment_request_system.approve_payment_request(
          request_id=1
      )
  )

  double_approval_test.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": false,
  "message": "Payment request has already been approved or rejected.",
  "payment_request": {
    "request_id": 1,
    "requester_id": 1,
    "payer_id": 2,
    "amount": 100.0,
    "status": "approved",
    "created_at": "2026-09-28 12:35:03.456267"
  }
}
Metadata:
{
  "action": "approve_payment_request",
  "status": "rejected",
  "request_id": 1
}
time: 1.49 ms (started: 2026-09-28 12:35:44 +00:00)


### Duplicate Payment Approval Validation

The validation confirms that the second approval attempt is rejected, no additional transaction is created, and the users' wallet balances remain unchanged.

In [70]:
if RUN_COMPONENT_TESTS:
  assert double_approval_test.output["success"] is False

  assert (
      payment_request_system.payment_requests[1].status
      == "approved"
  )

  assert len(payment_request_system.transactions) == 1

  assert (
      payment_request_system.wallets[1].balance
      == 900.0
  )

  assert (
      payment_request_system.wallets[2].balance
      == 400.0
  )

  print("Double-approval prevention test passed.")

Double-approval prevention test passed.
time: 825 µs (started: 2026-09-28 12:35:47 +00:00)


### Payment Request Rejection Test

This test creates a new payment request and then rejects it.

Rejecting the request updates its status without creating a transaction or modifying either user's wallet balance.

In [71]:
if RUN_COMPONENT_TESTS:
  second_payment_request = (
      payment_request_system.request_payment(
          requester_id=2,
          payer_id=1,
          amount=50.0
      )
  )

  reject_request_test = (
      payment_request_system.reject_payment_request(
          request_id=2
      )
  )

  reject_request_test.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "message": "Payment request rejected successfully.",
  "payment_request": {
    "request_id": 2,
    "requester_id": 2,
    "payer_id": 1,
    "amount": 50.0,
    "status": "rejected",
    "created_at": "2026-09-28 12:35:49.903777"
  }
}
Metadata:
{
  "action": "reject_payment_request",
  "status": "success",
  "request_id": 2
}
time: 875 µs (started: 2026-09-28 12:35:49 +00:00)


### Payment Request Rejection Validation

The validation confirms that the payment request status is updated to `rejected`, no additional transaction is created, and the users' wallet balances remain unchanged.


In [72]:
if RUN_COMPONENT_TESTS:
  assert second_payment_request.output["success"] is True
  assert reject_request_test.output["success"] is True

  assert (
      payment_request_system.payment_requests[2].status
      == "rejected"
  )

  assert len(payment_request_system.transactions) == 1

  assert (
      payment_request_system.wallets[1].balance
      == 900.0
  )

  assert (
      payment_request_system.wallets[2].balance
      == 400.0
  )

  print("Payment-request rejection test passed.")

Payment-request rejection test passed.
time: 2.93 ms (started: 2026-09-28 12:35:52 +00:00)


### Duplicate Payment Rejection Test

Similar to duplicate approval, a payment request that has already been rejected cannot be rejected again.

This test attempts to reject an already rejected payment request to verify that the system prevents duplicate state transitions.


In [73]:
if RUN_COMPONENT_TESTS:
  double_rejection_test = (
      payment_request_system.reject_payment_request(
          request_id=2
      )
  )

  double_rejection_test.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": false,
  "message": "Payment request has already been approved or rejected.",
  "payment_request": {
    "request_id": 2,
    "requester_id": 2,
    "payer_id": 1,
    "amount": 50.0,
    "status": "rejected",
    "created_at": "2026-09-28 12:35:49.903777"
  }
}
Metadata:
{
  "action": "reject_payment_request",
  "status": "rejected",
  "request_id": 2
}
time: 781 µs (started: 2026-09-28 12:35:55 +00:00)


### Duplicate Payment Rejection Validation

The validation confirms that the second rejection attempt is rejected and that the system state remains unchanged.

In [75]:
if RUN_COMPONENT_TESTS:
  assert double_rejection_test.output["success"] is False

  assert (
      payment_request_system.payment_requests[2].status
      == "rejected"
  )

  assert len(payment_request_system.transactions) == 1

  print("Double-rejection prevention test passed.")

Double-rejection prevention test passed.
time: 841 µs (started: 2026-09-28 12:36:32 +00:00)


# Short-Term Business Memory

The `BusinessMemory` component maintains contextual information about recent system activity and operates independently from the `PaymentSystem`.

Its responsibilities include:

- Tracking recent workflow activity.
- Supporting follow-up requests.
- Providing context for explanations of the most recent action or transaction.
- Sharing relevant context between agents.
- Storing the latest critic score.

The memory layer does not execute financial operations or modify wallet balances, transactions, or payment requests.

In [76]:
@dataclass
class BusinessMemory:
    last_intent: Optional[str] = None
    last_tool: Optional[str] = None
    last_user_message: Optional[str] = None
    last_action: Optional[str] = None
    last_user: Optional[Dict[str, Any]] = None
    last_transaction: Optional[Dict[str, Any]] = None
    last_payment_request: Optional[Dict[str, Any]] = None
    last_result: Optional[Dict[str, Any]] = None
    last_critic_score: Optional[float] = None
    recent_actions: Optional[List[str]] = None

    def __post_init__(self) -> None:
        if self.recent_actions is None:
            self.recent_actions = []

time: 6.74 ms (started: 2026-09-28 12:37:11 +00:00)


## Business Memory Updates

The `update` method selectively updates only the memory fields relevant to the latest operation.

If a field is not provided, its previously stored value is preserved, allowing contextual information to persist across related interactions.

Each new action is also added to `recent_actions`, which maintains a rolling history of the ten most recent operations.

In [77]:
def update_business_memory(
    memory: BusinessMemory,
    *,
    intent: Optional[str] = None,
    tool: Optional[str] = None,
    user_message: Optional[str] = None,
    action: Optional[str] = None,
    user: Optional[Dict[str, Any]] = None,
    transaction: Optional[Dict[str, Any]] = None,
    payment_request: Optional[Dict[str, Any]] = None,
    result: Optional[Dict[str, Any]] = None,
    critic_score: Optional[float] = None
) -> None:
    if intent is not None:
        memory.last_intent = intent

    if tool is not None:
        memory.last_tool = tool

    if user_message is not None:
        memory.last_user_message = user_message

    if action is not None:
        memory.last_action = action
        memory.recent_actions.append(action)
        memory.recent_actions = memory.recent_actions[-10:]

    if user is not None:
        memory.last_user = user

    if transaction is not None:
        memory.last_transaction = transaction

    if payment_request is not None:
        memory.last_payment_request = payment_request

    if result is not None:
        memory.last_result = result

    if critic_score is not None:
        memory.last_critic_score = critic_score

time: 1.93 ms (started: 2026-09-28 12:37:13 +00:00)


## Business Memory Reset

The `reset_business_memory` function restores the business memory to its initial state.

It is primarily used during testing to ensure that each test scenario begins with a clean, isolated memory state and does not depend on data produced by previous tests.

In [78]:
def reset_business_memory(
    memory: BusinessMemory
) -> None:
    memory.last_intent = None
    memory.last_tool = None
    memory.last_user_message = None
    memory.last_action = None
    memory.last_user = None
    memory.last_transaction = None
    memory.last_payment_request = None
    memory.last_result = None
    memory.last_critic_score = None
    memory.recent_actions = []

time: 4.02 ms (started: 2026-09-28 12:37:17 +00:00)


## Business Memory Display

The `display_business_memory` function provides a readable representation of the current business memory state.

It is intended for testing and demonstration purposes only and does not modify any stored memory values.

In [79]:
def display_business_memory(
    memory: BusinessMemory
) -> None:
    print("=" * 60)
    print("BUSINESS MEMORY")
    print("=" * 60)

    print(json.dumps(
        asdict(memory),
        indent=2,
        ensure_ascii=False,
        default=str
    ))

    print("=" * 60)

time: 758 µs (started: 2026-09-28 12:38:24 +00:00)


## Shared Business Memory Instance

A single shared `BusinessMemory` instance is created to maintain context across the multi-agent workflow.

The `OrchestratorAgent` updates this shared memory after relevant operations, while specialized agents access the contextual information they need to perform their respective tasks.

In [80]:
business_memory = BusinessMemory()

display_business_memory(
    business_memory
)

BUSINESS MEMORY
{
  "last_intent": null,
  "last_tool": null,
  "last_user_message": null,
  "last_action": null,
  "last_user": null,
  "last_transaction": null,
  "last_payment_request": null,
  "last_result": null,
  "last_critic_score": null,
  "recent_actions": []
}
time: 663 µs (started: 2026-09-28 12:38:26 +00:00)


### Action and User Memory Update Test

This test simulates a user-creation operation and updates the shared business memory with the resulting context.

The validation verifies that the memory correctly stores the intent, tool, user message, action, user information, and operation result.

In [81]:
if RUN_COMPONENT_TESTS:
  memory_user_example = {
      "user_id": 1,
      "name": "Alice",
      "phone_number": "0501234567"
  }

  memory_result_example = {
      "success": True,
      "message": "User created successfully."
  }

  update_business_memory(
      memory=business_memory,
      intent=Intent.CREATE_USER.value,
      tool="payment_system",
      user_message=(
          "Create a new user named Alice "
          "with phone 0501234567 and balance 800"
      ),
      action="create_user",
      user=memory_user_example,
      result=memory_result_example
  )

  display_business_memory(
      business_memory
  )

BUSINESS MEMORY
{
  "last_intent": "createUser",
  "last_tool": "payment_system",
  "last_user_message": "Create a new user named Alice with phone 0501234567 and balance 800",
  "last_action": "create_user",
  "last_user": {
    "user_id": 1,
    "name": "Alice",
    "phone_number": "0501234567"
  },
  "last_transaction": null,
  "last_payment_request": null,
  "last_result": {
    "success": true,
    "message": "User created successfully."
  },
  "last_critic_score": null,
  "recent_actions": [
    "create_user"
  ]
}
time: 1.01 ms (started: 2026-09-28 12:38:46 +00:00)


### User Memory Update Validation

The validation confirms that all relevant information associated with the user-creation operation is correctly stored in the business memory.

In [82]:
if RUN_COMPONENT_TESTS:
  assert (
      business_memory.last_intent
      == Intent.CREATE_USER.value
  )

  assert (
      business_memory.last_tool
      == "payment_system"
  )

  assert (
      business_memory.last_action
      == "create_user"
  )

  assert (
      business_memory.last_user["user_id"]
      == 1
  )

  assert (
      business_memory.last_result["success"]
      is True
  )

  assert (
      business_memory.recent_actions
      == ["create_user"]
  )

  print("BusinessMemory user-update test passed.")

BusinessMemory user-update test passed.
time: 994 µs (started: 2026-09-28 12:52:51 +00:00)


### Last Transaction Memory Test

This test executes a payment operation and stores the resulting transaction in the shared business memory.

The stored transaction provides contextual information that can later be used by the `ExplanationAgent` to explain the most recent transaction.


In [83]:
if RUN_COMPONENT_TESTS:
  memory_payment_system = PaymentSystem()

  memory_payment_system.create_user(
      name="Alice",
      phone_number="0501234567",
      initial_balance=800.0
  )

  memory_payment_system.create_user(
      name="Bob",
      phone_number="0507654321",
      initial_balance=500.0
  )

  memory_transfer_result = (
      memory_payment_system.transfer_money(
          sender_id=1,
          receiver_id=2,
          amount=200.0
      )
  )

  update_business_memory(
      memory=business_memory,
      intent=Intent.TRANSFER_MONEY.value,
      tool="payment_system",
      user_message="Transfer 200 from user 1 to user 2",
      action="transfer_money",
      transaction=memory_transfer_result.output[
          "transaction"
      ],
      result=memory_transfer_result.output
  )

  display_business_memory(
      business_memory
  )

BUSINESS MEMORY
{
  "last_intent": "transferMoney",
  "last_tool": "payment_system",
  "last_user_message": "Transfer 200 from user 1 to user 2",
  "last_action": "transfer_money",
  "last_user": {
    "user_id": 1,
    "name": "Alice",
    "phone_number": "0501234567"
  },
  "last_transaction": {
    "transaction_id": 1,
    "sender_id": 1,
    "receiver_id": 2,
    "amount": 200.0,
    "timestamp": "2026-09-28 12:52:54.338669",
    "status": "completed",
    "risk_score": 0.0
  },
  "last_payment_request": null,
  "last_result": {
    "success": true,
    "message": "Transfer completed successfully.",
    "transaction": {
      "transaction_id": 1,
      "sender_id": 1,
      "receiver_id": 2,
      "amount": 200.0,
      "timestamp": "2026-09-28 12:52:54.338669",
      "status": "completed",
      "risk_score": 0.0
    },
    "sender_balance": 600.0,
    "receiver_balance": 700.0
  },
  "last_critic_score": null,
  "recent_actions": [
    "create_user",
    "transfer_money"
  ]
}
ti

### Last Transaction Memory Validation

The validation confirms that the business memory correctly stores the most recent transaction and action, and that `recent_actions` contains both operations performed during the test.

In [84]:
if RUN_COMPONENT_TESTS:
  assert (
      business_memory.last_intent
      == Intent.TRANSFER_MONEY.value
  )

  assert (
      business_memory.last_action
      == "transfer_money"
  )

  assert (
      business_memory.last_transaction[
          "transaction_id"
      ]
      == 1
  )

  assert (
      business_memory.last_transaction["amount"]
      == 200.0
  )

  assert (
      business_memory.last_transaction["sender_id"]
      == 1
  )

  assert (
      business_memory.last_transaction["receiver_id"]
      == 2
  )

  assert business_memory.recent_actions == [
      "create_user",
      "transfer_money"
  ]

  print("BusinessMemory transaction test passed.")

BusinessMemory transaction test passed.
time: 1.03 ms (started: 2026-09-28 12:52:57 +00:00)


### Last Payment Request Memory Test

This test creates a payment request and stores it in the shared business memory.

The stored request provides contextual information that can later be used by the `ExplanationAgent` to explain the most recent payment request.

In [85]:
if RUN_COMPONENT_TESTS:
  memory_payment_request_result = (
      memory_payment_system.request_payment(
          requester_id=2,
          payer_id=1,
          amount=50.0
      )
  )

  update_business_memory(
      memory=business_memory,
      intent=Intent.REQUEST_PAYMENT.value,
      tool="payment_system",
      user_message=(
          "Request 50 from user 1 for user 2"
      ),
      action="request_payment",
      payment_request=(
          memory_payment_request_result
          .output["payment_request"]
      ),
      result=memory_payment_request_result.output
  )

  display_business_memory(
      business_memory
  )

BUSINESS MEMORY
{
  "last_intent": "requestPayment",
  "last_tool": "payment_system",
  "last_user_message": "Request 50 from user 1 for user 2",
  "last_action": "request_payment",
  "last_user": {
    "user_id": 1,
    "name": "Alice",
    "phone_number": "0501234567"
  },
  "last_transaction": {
    "transaction_id": 1,
    "sender_id": 1,
    "receiver_id": 2,
    "amount": 200.0,
    "timestamp": "2026-09-28 12:52:54.338669",
    "status": "completed",
    "risk_score": 0.0
  },
  "last_payment_request": {
    "request_id": 1,
    "requester_id": 2,
    "payer_id": 1,
    "amount": 50.0,
    "status": "pending",
    "created_at": "2026-09-28 12:53:00.797147"
  },
  "last_result": {
    "success": true,
    "message": "Payment request created successfully.",
    "payment_request": {
      "request_id": 1,
      "requester_id": 2,
      "payer_id": 1,
      "amount": 50.0,
      "status": "pending",
      "created_at": "2026-09-28 12:53:00.797147"
    }
  },
  "last_critic_score": n

### Last Payment Request Memory Validation

The validation confirms that the business memory correctly stores the most recent payment request and its current status.

In [86]:
if RUN_COMPONENT_TESTS:
  assert (
      business_memory.last_payment_request[
          "request_id"
      ]
      == 1
  )

  assert (
      business_memory.last_payment_request[
          "requester_id"
      ]
      == 2
  )

  assert (
      business_memory.last_payment_request[
          "payer_id"
      ]
      == 1
  )

  assert (
      business_memory.last_payment_request[
          "status"
      ]
      == "pending"
  )

  assert business_memory.recent_actions == [
      "create_user",
      "transfer_money",
      "request_payment"
  ]

  print("BusinessMemory payment-request test passed.")

BusinessMemory payment-request test passed.
time: 907 µs (started: 2026-09-28 12:53:05 +00:00)


### Recent Actions Limit Test

The business memory stores a maximum of ten recent actions.

This test adds additional actions and verifies that the `recent_actions` list does not exceed its defined limit, ensuring that only the most recent activity is retained.


In [87]:
if RUN_COMPONENT_TESTS:
  for index in range(12):
      update_business_memory(
          memory=business_memory,
          action=f"test_action_{index}"
      )

  assert len(
      business_memory.recent_actions
  ) == 10

  assert (
      business_memory.recent_actions[0]
      == "test_action_2"
  )

  assert (
      business_memory.recent_actions[-1]
      == "test_action_11"
  )

  print("BusinessMemory recent-actions limit test passed.")

BusinessMemory recent-actions limit test passed.
time: 2.03 ms (started: 2026-09-28 12:53:09 +00:00)


### Business Memory Reset Test

At the end of the validation process, the business memory is reset to ensure that all stored fields return to their initial state.

This test verifies that no contextual data from previous operations remains after the reset.

In [88]:
if RUN_COMPONENT_TESTS:
  reset_business_memory(
      business_memory
  )

  assert business_memory.last_intent is None
  assert business_memory.last_tool is None
  assert business_memory.last_user_message is None
  assert business_memory.last_action is None
  assert business_memory.last_user is None
  assert business_memory.last_transaction is None
  assert business_memory.last_payment_request is None
  assert business_memory.last_result is None
  assert business_memory.last_critic_score is None
  assert business_memory.recent_actions == []

  display_business_memory(
      business_memory
  )

  print("BusinessMemory reset test passed.")

BUSINESS MEMORY
{
  "last_intent": null,
  "last_tool": null,
  "last_user_message": null,
  "last_action": null,
  "last_user": null,
  "last_transaction": null,
  "last_payment_request": null,
  "last_result": null,
  "last_critic_score": null,
  "recent_actions": []
}
BusinessMemory reset test passed.
time: 1.18 ms (started: 2026-09-28 12:53:11 +00:00)


# Fallback Agent

The `FallbackAgent` handles requests that cannot be safely or reliably processed by the standard workflow.

It may be triggered when:

- The `RouterAgent` returns an `unknown` intent.
- Required parameters are missing from the request.
- The routing confidence is below the required threshold.
- The `CriticAgent` determines that a result does not meet the required quality criteria.

The agent uses an LLM to generate a clear response but does not execute financial operations or modify the system state.

Its responsibility is to explain why the request cannot be completed and identify any missing information required to continue.

## FallbackAgent Structured Output

The `FallbackAgent` returns a structured response containing:

- A clear message for the user.
- The reason the fallback mechanism was triggered.
- A list of missing or required information.
- A suggestion for how the request can be reformulated or completed.

In [89]:
class FallbackOutput(BaseModel):
    message: str = Field(
        description="A clear message explaining why the request cannot continue."
    )

    reason: str = Field(
        description="The reason the fallback mechanism was activated."
    )

    missing_fields: List[str] = Field(
        default_factory=list,
        description="Required fields missing from the request."
    )

    suggested_request: str = Field(
        description="A corrected example of how the user can rewrite the request."
    )

time: 4.41 ms (started: 2026-09-28 12:53:16 +00:00)


## FallbackAgent Instructions

The `FallbackAgent` receives the original user request, the detected intent, and any missing required parameters.

It is instructed to generate a concise and clear response explaining what information is needed to continue. The agent must not fabricate user IDs, transaction amounts, or other user information that was not explicitly provided.

In [90]:
FALLBACK_SYSTEM_PROMPT = """
You are the FallbackAgent of an educational digital payment system.

Your responsibilities are:

1. Explain why the request cannot be completed.
2. Identify missing information.
3. Suggest a clearer version of the request.

Rules:

- Do not perform financial operations.
- Do not invent user identifiers.
- Do not invent payment-request identifiers.
- Do not invent amounts.
- Do not claim that an operation succeeded.
- Keep the response short and practical.
- Use numeric user identifiers for financial operations.
- If the intent is unknown, explain that the requested action was unclear.
"""

time: 641 µs (started: 2026-09-28 12:53:18 +00:00)


## FallbackAgent Implementation

The `FallbackAgent` uses the configured OpenAI provider to generate recovery guidance for requests that cannot proceed through the standard workflow.

It receives structured context from the system, including the original request, detected intent, and missing parameters, and returns a structured response that helps the user correct or complete the request.

In [91]:
class FallbackAgent:
    def __init__(
        self,
        provider: LLMProvider
    ):
        self.provider = provider

    def run(
        self,
        user_message: str,
        reason: str,
        intent: Optional[str] = None,
        missing_fields: Optional[List[str]] = None
    ) -> AgentResult:
        missing_fields = missing_fields or []

        fallback_context = {
            "user_message": user_message,
            "detected_intent": intent,
            "reason": reason,
            "missing_fields": missing_fields
        }

        result = self.provider.generate_json(
            system_prompt=FALLBACK_SYSTEM_PROMPT,
            user_prompt=json.dumps(
                fallback_context,
                ensure_ascii=False,
                indent=2
            ),
            output_model=FallbackOutput
        )

        return AgentResult(
            agent_name="FallbackAgent",
            output=result.model_dump(),
            confidence=1.0,
            metadata={
                "provider": self.provider.provider_name,
                "model": self.provider.model_name,
                "status": "fallback",
                "detected_intent": intent
            }
        )

time: 3.02 ms (started: 2026-09-28 12:53:20 +00:00)


## FallbackAgent Initialization

The `FallbackAgent` is initialized with the configured OpenAI provider.

In [92]:
fallback_agent = FallbackAgent(
    openai_provider
)

time: 365 µs (started: 2026-09-28 12:53:24 +00:00)


### FallbackAgent Missing Parameters Test

This test evaluates how the `FallbackAgent` handles a money-transfer request with missing sender and receiver IDs.

The `RouterAgent` may correctly identify the `transferMoney` intent, but the operation cannot be executed without the required numeric user identifiers.

The `FallbackAgent` is therefore expected to explain the missing information and guide the user toward completing the request.

In [93]:
if RUN_COMPONENT_TESTS:
  fallback_missing_fields_test = fallback_agent.run(
      user_message="Transfer 150 from Alice to Bob",
      reason="Required action parameters are missing.",
      intent=Intent.TRANSFER_MONEY.value,
      missing_fields=[
          "sender_id",
          "receiver_id"
      ]
  )

  fallback_missing_fields_test.display()

Agent name: FallbackAgent
Confidence: 1.0
Output:
{
  "message": "I cannot complete the transfer because the required numeric sender_id and receiver_id are missing. I can’t infer or invent user IDs from names like “Alice” and “Bob,” and I won’t perform financial operations without explicit identifiers.",
  "reason": "Required action parameters are missing.",
  "missing_fields": [
    "sender_id",
    "receiver_id"
  ],
  "suggested_request": "Example: {\"detected_intent\":\"transferMoney\",\"amount\":150,\"currency\":\"USD\",\"sender_id\":<numeric_sender_id>,\"receiver_id\":<numeric_receiver_id>} — replace the placeholders with the actual numeric user IDs."
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "fallback",
  "detected_intent": "transferMoney"
}
time: 11.1 s (started: 2026-09-28 12:53:26 +00:00)


In [94]:
if RUN_COMPONENT_TESTS:
  assert (
      fallback_missing_fields_test.agent_name
      == "FallbackAgent"
  )

  assert (
      fallback_missing_fields_test.metadata["status"]
      == "fallback"
  )

  assert (
      fallback_missing_fields_test.metadata["detected_intent"]
      == Intent.TRANSFER_MONEY.value
  )

  assert (
      "sender_id"
      in fallback_missing_fields_test.output["missing_fields"]
  )

  assert (
      "receiver_id"
      in fallback_missing_fields_test.output["missing_fields"]
  )

  print("FallbackAgent missing-fields test passed.")

FallbackAgent missing-fields test passed.
time: 1.02 ms (started: 2026-09-28 12:53:40 +00:00)


# Explanation Agent

The `ExplanationAgent` generates user-friendly explanations of the most recent operation stored in the shared business memory.

It uses `BusinessMemory` to access contextual information such as:

- The most recent action.
- The most recently referenced user.
- The most recent transaction.
- The most recent payment request.
- The result of the most recent operation.

The agent uses this context to explain completed system activity without modifying the business memory or executing any financial operation.


## ExplanationAgent Structured Output

The `ExplanationAgent` returns a structured response containing:

- The type of explanation generated.
- A concise, user-friendly explanation.
- Whether the referenced operation was successful.
- The action or operation being explained.

In [95]:
class ExplanationOutput(BaseModel):
    explanation_type: str = Field(
        description=(
            "The type of explanation: transaction, payment_request, "
            "action or no_context."
        )
    )

    explanation: str = Field(
        description="A clear explanation of the last recorded operation."
    )

    action: Optional[str] = Field(
        default=None,
        description="The last business action."
    )

    success: Optional[bool] = Field(
        default=None,
        description="Whether the last operation succeeded."
    )

time: 2.77 ms (started: 2026-09-28 12:53:49 +00:00)


## ExplanationAgent Instructions

The `ExplanationAgent` must generate explanations using only information available in the shared business memory.

It must not fabricate transactions, payment requests, amounts, users, risk scores, or any other information that is not present in the stored context.

This constraint ensures that explanations remain grounded in actual system activity.

In [96]:
EXPLANATION_SYSTEM_PROMPT = """
You are the ExplanationAgent of an educational digital payment system.

Your responsibility is to explain the last business operation
using only the provided business memory.

Priority rules:

1. Always explain the most recent business action using last_action and last_result.
2. If last_action is transfer_money and last_result contains a transaction,
   explain that transaction.
3. If last_action is request_payment, approve_payment_request,
   or reject_payment_request and last_result contains a payment_request,
   explain that payment request.
4. If the last action failed, explain the rejection using last_result.
5. Use last_transaction or last_payment_request only as supporting context.
6. If no action context exists, return explanation_type=no_context.


Rules:

- Do not invent information.
- Do not change transaction or payment-request data.
- Do not claim an operation succeeded unless the memory says it succeeded.
- Mention sender_id, receiver_id and amount when explaining a transaction.
- Mention requester_id, payer_id, amount and status when explaining a payment request.
- If the last operation was rejected, explain the rejection reason from last_result.
- Do not explain an older transaction when last_action refers to a newer failed operation.
- Mention the risk score only if it exists in the transaction data.
- Keep the explanation clear and concise.
"""

time: 1 ms (started: 2026-09-28 12:58:49 +00:00)


## ExplanationAgent Implementation

The `ExplanationAgent` uses the configured OpenAI provider to generate structured, user-friendly explanations.

It receives a structured snapshot of the current `BusinessMemory` as context and returns its explanation using the predefined output schema.

This design keeps explanation generation separate from both business-state management and financial execution.

In [97]:
class ExplanationAgent:
    def __init__(
        self,
        provider: LLMProvider
    ):
        self.provider = provider

    def run(
        self,
        memory: BusinessMemory
    ) -> AgentResult:
        memory_data = asdict(memory)

        result = self.provider.generate_json(
            system_prompt=EXPLANATION_SYSTEM_PROMPT,
            user_prompt=json.dumps(
                memory_data,
                ensure_ascii=False,
                indent=2,
                default=str
            ),
            output_model=ExplanationOutput
        )

        return AgentResult(
            agent_name="ExplanationAgent",
            output=result.model_dump(
                exclude_none=True
            ),
            confidence=0.95,
            metadata={
                "provider": self.provider.provider_name,
                "model": self.provider.model_name,
                "status": "success",
                "last_action": memory.last_action
            }
        )

time: 2.67 ms (started: 2026-09-28 12:58:53 +00:00)


## ExplanationAgent Initialization

The `ExplanationAgent` is initialized with the configured OpenAI provider.

A focused validation is performed to verify that the agent can generate a structured explanation from the information stored in `BusinessMemory`.


In [98]:
explanation_agent = ExplanationAgent(
    openai_provider
)

time: 391 µs (started: 2026-09-28 12:58:57 +00:00)


###Tests


### ExplanationAgent Transaction Context Test

This test creates a fresh payment system, executes a money transfer, and stores the resulting transaction in `BusinessMemory`.

The `ExplanationAgent` then uses only the stored memory context to generate an explanation of the most recent transaction.

In [99]:
if RUN_COMPONENT_TESTS:
  explanation_test_system = PaymentSystem()

  explanation_test_system.create_user(
      name="Alice",
      phone_number="0501234567",
      initial_balance=800.0
  )

  explanation_test_system.create_user(
      name="Bob",
      phone_number="0507654321",
      initial_balance=500.0
  )

  explanation_transfer_result = (
      explanation_test_system.transfer_money(
          sender_id=1,
          receiver_id=2,
          amount=200.0
      )
  )

  reset_business_memory(
      business_memory
  )

  update_business_memory(
      memory=business_memory,
      intent=Intent.TRANSFER_MONEY.value,
      tool="payment_system",
      user_message="Transfer 200 from user 1 to user 2",
      action="transfer_money",
      transaction=explanation_transfer_result.output[
          "transaction"
      ],
      result=explanation_transfer_result.output
  )

  display_business_memory(
      business_memory
  )

BUSINESS MEMORY
{
  "last_intent": "transferMoney",
  "last_tool": "payment_system",
  "last_user_message": "Transfer 200 from user 1 to user 2",
  "last_action": "transfer_money",
  "last_user": null,
  "last_transaction": {
    "transaction_id": 1,
    "sender_id": 1,
    "receiver_id": 2,
    "amount": 200.0,
    "timestamp": "2026-09-28 12:59:00.294509",
    "status": "completed",
    "risk_score": 0.0
  },
  "last_payment_request": null,
  "last_result": {
    "success": true,
    "message": "Transfer completed successfully.",
    "transaction": {
      "transaction_id": 1,
      "sender_id": 1,
      "receiver_id": 2,
      "amount": 200.0,
      "timestamp": "2026-09-28 12:59:00.294509",
      "status": "completed",
      "risk_score": 0.0
    },
    "sender_balance": 600.0,
    "receiver_balance": 700.0
  },
  "last_critic_score": null,
  "recent_actions": [
    "transfer_money"
  ]
}
time: 1.59 ms (started: 2026-09-28 12:59:00 +00:00)


### Last Transaction Explanation Test

The `ExplanationAgent` receives the business memory after the transfer and generates a structured explanation of the most recent transaction using the stored context.

In [100]:
if RUN_COMPONENT_TESTS:
  explanation_transaction_test = (
      explanation_agent.run(
          business_memory
      )
  )

  explanation_transaction_test.display()

Agent name: ExplanationAgent
Confidence: 0.95
Output:
{
  "explanation_type": "transaction",
  "explanation": "Transaction 1: transfer of 200.0 from sender_id 1 to receiver_id 2 completed. Status: completed. Risk score: 0.0. Sender balance: 600.0. Receiver balance: 700.0.",
  "action": "transfer_money",
  "success": true
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "success",
  "last_action": "transfer_money"
}
time: 7.84 s (started: 2026-09-28 12:59:05 +00:00)


In [101]:
if RUN_COMPONENT_TESTS:
  assert (
      explanation_transaction_test.agent_name
      == "ExplanationAgent"
  )

  assert (
      explanation_transaction_test.output[
          "explanation_type"
      ]
      == "transaction"
  )

  explanation_text = (
      explanation_transaction_test.output[
          "explanation"
      ]
  )

  assert "1" in explanation_text
  assert "2" in explanation_text
  assert "200" in explanation_text

  assert (
      explanation_transaction_test.metadata[
          "last_action"
      ]
      == "transfer_money"
  )

  print("ExplanationAgent transaction test passed.")

ExplanationAgent transaction test passed.
time: 937 µs (started: 2026-09-28 12:59:20 +00:00)


# Fraud Detection Agent

The `FraudDetectionAgent` evaluates completed money transfers for potentially suspicious activity.

The fraud assessment considers several transaction-level factors:

- The transaction amount.
- The ratio between the transaction amount and the sender's balance before the transfer.
- The number of transactions previously performed by the sender.
- A baseline risk score calculated by deterministic Python logic.

Numerical risk features are calculated by deterministic Python tools and then provided to the LLM agent, which produces a structured risk assessment and explanation.

The `FraudDetectionAgent` performs analysis only. It does not reverse transactions, modify wallet balances, or directly alter the financial state of the system.

## Deterministic Risk Feature Calculator

The `FraudFeatureCalculator` computes transaction risk features using deterministic Python logic.

The numerical thresholds are explicitly defined as configurable parameters, making the risk calculation transparent and easy to adjust or evaluate.

The calculator does not act as an autonomous agent and does not independently determine whether a transaction is fraudulent. Instead, it prepares structured risk features that are passed to the `FraudDetectionAgent` for further assessment.


In [102]:
class FraudFeatureCalculator:
    def __init__(
        self,
        high_amount_threshold: float = 500.0,
        high_balance_ratio: float = 0.80,
        frequent_transaction_threshold: int = 3
    ):
        self.high_amount_threshold = high_amount_threshold
        self.high_balance_ratio = high_balance_ratio
        self.frequent_transaction_threshold = (
            frequent_transaction_threshold
        )

    def calculate(
        self,
        transaction: Dict[str, Any],
        sender_balance_before: float,
        sender_transaction_count: int
    ) -> Dict[str, Any]:
        amount = float(transaction["amount"])

        if sender_balance_before > 0:
            balance_ratio = amount / sender_balance_before
        else:
            balance_ratio = 1.0

        amount_flag = (
            amount >= self.high_amount_threshold
        )

        balance_ratio_flag = (
            balance_ratio >= self.high_balance_ratio
        )

        frequency_flag = (
            sender_transaction_count
            >= self.frequent_transaction_threshold
        )

        base_risk_score = 0.0

        if amount_flag:
            base_risk_score += 0.40

        if balance_ratio_flag:
            base_risk_score += 0.35

        if frequency_flag:
            base_risk_score += 0.25

        base_risk_score = round(
            min(base_risk_score, 1.0),
            2
        )

        return {
            "transaction": transaction,
            "sender_balance_before": sender_balance_before,
            "sender_transaction_count": sender_transaction_count,
            "balance_ratio": round(balance_ratio, 3),
            "amount_flag": amount_flag,
            "balance_ratio_flag": balance_ratio_flag,
            "frequency_flag": frequency_flag,
            "base_risk_score": base_risk_score,
            "configured_thresholds": {
                "high_amount_threshold": (
                    self.high_amount_threshold
                ),
                "high_balance_ratio": (
                    self.high_balance_ratio
                ),
                "frequent_transaction_threshold": (
                    self.frequent_transaction_threshold
                )
            }
        }

time: 2.58 ms (started: 2026-09-28 12:59:26 +00:00)


## FraudDetectionAgent Structured Output

The `FraudDetectionAgent` returns a structured risk assessment containing:

- Whether the transaction is classified as suspicious.
- The assessed risk level.
- A numerical risk score.
- The factors contributing to the assessment.
- A recommendation for further review.

In [103]:
class FraudDetectionOutput(BaseModel):
    suspicious: bool = Field(
        description="Whether the transaction should be considered suspicious."
    )

    risk_level: str = Field(
        description="The risk level: low, medium or high."
    )

    risk_score: float = Field(
        ge=0.0,
        le=1.0,
        description="The final transaction risk score."
    )

    reasons: List[str] = Field(
        default_factory=list,
        description="Reasons supporting the risk decision."
    )

    recommended_action: str = Field(
        description="A recommendation for the next system step."
    )

time: 6.82 ms (started: 2026-09-28 12:59:31 +00:00)


## FraudDetectionAgent Structured Output

The `FraudDetectionAgent` returns a structured risk assessment containing:

- Whether the transaction is classified as suspicious.
- The assessed risk level.
- A numerical risk score.
- The factors contributing to the assessment.
- A recommendation for further review.


In [104]:
FRAUD_SYSTEM_PROMPT = """
You are the FraudDetectionAgent of an educational payment system.

Analyze the provided transaction-risk features.

Your responsibilities are:

1. Decide whether the transaction is suspicious.
2. Return a risk level: low, medium or high.
3. Return a risk score between 0 and 1.
4. Explain which provided factors affected the decision.
5. Recommend whether the transaction requires additional security review.

Rules:

- Use only the supplied data.
- Do not invent previous transactions.
- Do not invent balances.
- Do not change the transaction.
- Do not claim that the transaction was blocked.
- The transaction has already been completed.
- Treat the provided base_risk_score as the numerical foundation.
- Keep the final risk_score consistent with the provided evidence.
- If the risk is medium or high, recommend a security review.
"""

time: 1.03 ms (started: 2026-09-28 12:59:35 +00:00)


## FraudDetectionAgent Implementation

The `FraudDetectionAgent` uses the configured OpenAI provider to analyze the risk features produced by the deterministic calculation layer.

It processes the structured risk context and returns its assessment using the standardized `AgentResult` format, allowing the result to be consumed consistently by downstream components.

In [105]:
class FraudDetectionAgent:
    def __init__(
        self,
        provider: LLMProvider,
        feature_calculator: FraudFeatureCalculator
    ):
        self.provider = provider
        self.feature_calculator = feature_calculator

    def run(
        self,
        transaction: Dict[str, Any],
        sender_balance_before: float,
        sender_transaction_count: int
    ) -> AgentResult:
        fraud_features = self.feature_calculator.calculate(
            transaction=transaction,
            sender_balance_before=sender_balance_before,
            sender_transaction_count=sender_transaction_count
        )

        result = self.provider.generate_json(
            system_prompt=FRAUD_SYSTEM_PROMPT,
            user_prompt=json.dumps(
                fraud_features,
                ensure_ascii=False,
                indent=2,
                default=str
            ),
            output_model=FraudDetectionOutput
        )

        return AgentResult(
            agent_name="FraudDetectionAgent",
            output=result.model_dump(),
            confidence=0.95,
            metadata={
                "provider": self.provider.provider_name,
                "model": self.provider.model_name,
                "status": "success",
                "transaction_id": transaction[
                    "transaction_id"
                ],
                "fraud_features": fraud_features
            }
        )

time: 1.39 ms (started: 2026-09-28 12:59:38 +00:00)


## FraudDetectionAgent Initialization

The deterministic `FraudFeatureCalculator` and the `FraudDetectionAgent` are initialized to support the fraud-analysis workflow.

The calculator prepares the transaction risk features, while the agent uses the configured OpenAI provider to generate the structured risk assessment.

In [106]:
fraud_feature_calculator = FraudFeatureCalculator()

fraud_detection_agent = FraudDetectionAgent(
    provider=openai_provider,
    feature_calculator=fraud_feature_calculator
)

time: 766 µs (started: 2026-09-28 12:59:42 +00:00)


<div dir="rtl">

## Tests

</div>

### Standard Transaction Test

This test creates a transaction with an amount that is low relative to the sender's available balance.

The transaction is expected to remain below all risk thresholds defined by the `FraudFeatureCalculator`, providing a baseline case for evaluating the `FraudDetectionAgent`.

In [107]:
if RUN_COMPONENT_TESTS:
  normal_fraud_system = PaymentSystem()

  normal_fraud_system.create_user(
      name="Alice",
      phone_number="0501234567",
      initial_balance=1000.0
  )

  normal_fraud_system.create_user(
      name="Bob",
      phone_number="0507654321",
      initial_balance=500.0
  )

  normal_transfer_result = (
      normal_fraud_system.transfer_money(
          sender_id=1,
          receiver_id=2,
          amount=100.0
      )
  )

  normal_fraud_result = fraud_detection_agent.run(
      transaction=normal_transfer_result.output[
          "transaction"
      ],
      sender_balance_before=normal_transfer_result.metadata[
          "sender_balance_before"
      ],
      sender_transaction_count=1
  )

  normal_fraud_result.display()

Agent name: FraudDetectionAgent
Confidence: 0.95
Output:
{
  "suspicious": false,
  "risk_level": "low",
  "risk_score": 0.0,
  "reasons": [
    "Base risk score is 0.0 and used as the foundation for the final score.",
    "Transaction amount (100.0) is below the configured high_amount_threshold (500.0).",
    "Balance ratio (0.1) is well below the configured high_balance_ratio (0.8).",
    "Sender transaction count (1) is below the frequent_transaction_threshold (3).",
    "All indicator flags are false: amount_flag, balance_ratio_flag, frequency_flag."
  ],
  "recommended_action": "No additional security review required (low risk)."
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "success",
  "transaction_id": 1,
  "fraud_features": {
    "transaction": {
      "transaction_id": 1,
      "sender_id": 1,
      "receiver_id": 2,
      "amount": 100.0,
      "timestamp": "2026-09-28 12:59:44.953566",
      "status": "completed",
      "risk_score": 0.0
    },


In [108]:
if RUN_COMPONENT_TESTS:
  normal_features = (
      normal_fraud_result.metadata[
          "fraud_features"
      ]
  )

  assert normal_features["amount_flag"] is False
  assert normal_features["balance_ratio_flag"] is False
  assert normal_features["frequency_flag"] is False
  assert normal_features["base_risk_score"] == 0.0

  assert (
      normal_fraud_result.output["risk_level"]
      in ["low", "medium", "high"]
  )

  assert (
      0.0
      <= normal_fraud_result.output["risk_score"]
      <= 1.0
  )

  print("FraudDetectionAgent normal-transaction test passed.")

FraudDetectionAgent normal-transaction test passed.
time: 1.12 ms (started: 2026-09-28 13:00:00 +00:00)


### Suspicious Transaction Test

This test creates a high-value transaction that represents a significant portion of the sender's available balance.

The agent is also provided with information indicating that the sender has already performed multiple transactions.

Under these conditions, all three risk indicators defined by the `FraudFeatureCalculator` are expected to be triggered.

In [109]:
if RUN_COMPONENT_TESTS:
  suspicious_fraud_system = PaymentSystem()

  suspicious_fraud_system.create_user(
      name="Alice",
      phone_number="0501234567",
      initial_balance=1000.0
  )

  suspicious_fraud_system.create_user(
      name="Bob",
      phone_number="0507654321",
      initial_balance=100.0
  )

  suspicious_transfer_result = (
      suspicious_fraud_system.transfer_money(
          sender_id=1,
          receiver_id=2,
          amount=850.0
      )
  )

  suspicious_fraud_result = fraud_detection_agent.run(
      transaction=suspicious_transfer_result.output[
          "transaction"
      ],
      sender_balance_before=(
          suspicious_transfer_result.metadata[
              "sender_balance_before"
          ]
      ),
      sender_transaction_count=3
  )

  suspicious_fraud_result.display()

Agent name: FraudDetectionAgent
Confidence: 0.95
Output:
{
  "suspicious": true,
  "risk_level": "high",
  "risk_score": 1.0,
  "reasons": [
    "Base risk score provided is 1.0 and used as the numerical foundation.",
    "Transaction amount (850.0) exceeds configured high_amount_threshold (500.0). (amount_flag = true)",
    "Sender's balance ratio after the transaction is 0.85, which exceeds the configured high_balance_ratio (0.8), indicating a large portion of funds was moved. (balance_ratio_flag = true)",
    "Sender transaction count (3) meets the configured frequent_transaction_threshold (3), indicating elevated frequency of transactions. (frequency_flag = true)"
  ],
  "recommended_action": "Require additional security review (manual review / fraud investigation) because the transaction meets multiple high-risk conditions."
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "success",
  "transaction_id": 1,
  "fraud_features": {
    "transaction": {
      

In [110]:
if RUN_COMPONENT_TESTS:
  suspicious_features = (
      suspicious_fraud_result.metadata[
          "fraud_features"
      ]
  )

  assert suspicious_features["amount_flag"] is True
  assert suspicious_features["balance_ratio_flag"] is True
  assert suspicious_features["frequency_flag"] is True
  assert suspicious_features["base_risk_score"] == 1.0

  assert (
      suspicious_fraud_result.output["suspicious"]
      is True
  )

  assert (
      suspicious_fraud_result.output["risk_level"]
      in ["medium", "high"]
  )

  assert (
      suspicious_fraud_result.output["risk_score"]
      >= 0.70
  )

  print("FraudDetectionAgent suspicious-transaction test passed.")

FraudDetectionAgent suspicious-transaction test passed.
time: 1.1 ms (started: 2026-09-28 13:00:16 +00:00)


# Security Agent

The `SecurityAgent` evaluates the security mechanisms and business-rule protections implemented within the payment system.

The security review focuses on several core safeguards:

- Verifying that an audit log is maintained.
- Confirming that completed transfers are recorded in the transaction history.
- Verifying that the required payment operations are available.
- Checking that safeguards exist to prevent invalid or unauthorized operations.

Technical checks are performed deterministically by Python tools. The resulting security report is then analyzed by the LLM-based `SecurityAgent`, which produces a structured security assessment.

## Security Inspection Tool

The `SecurityInspectionTool` performs deterministic checks on the structure of the `PaymentSystem` and evaluates the results of its business-rule validations.

The tool is not an autonomous agent and does not modify the system state. Its responsibility is to collect and organize security-related findings into a structured report that can be analyzed by the `SecurityAgent`.

In [111]:
class SecurityInspectionTool:
    def inspect(
        self,
        payment_system: PaymentSystem
    ) -> Dict[str, Any]:
        required_attributes = [
            "users",
            "wallets",
            "transactions",
            "payment_requests",
            "audit_log"
        ]

        required_methods = [
            "create_user",
            "get_balance",
            "transfer_money",
            "get_transactions",
            "request_payment",
            "approve_payment_request",
            "reject_payment_request"
        ]

        attribute_checks = {
            attribute: hasattr(
                payment_system,
                attribute
            )
            for attribute in required_attributes
        }

        method_checks = {
            method: callable(
                getattr(
                    payment_system,
                    method,
                    None
                )
            )
            for method in required_methods
        }

        security_test_system = PaymentSystem()

        security_test_system.create_user(
            name="Alice",
            phone_number="0501234567",
            initial_balance=100.0
        )

        security_test_system.create_user(
            name="Bob",
            phone_number="0507654321",
            initial_balance=100.0
        )

        negative_amount_result = (
            security_test_system.transfer_money(
                sender_id=1,
                receiver_id=2,
                amount=-10.0
            )
        )

        self_transfer_result = (
            security_test_system.transfer_money(
                sender_id=1,
                receiver_id=1,
                amount=10.0
            )
        )

        missing_user_result = (
            security_test_system.transfer_money(
                sender_id=1,
                receiver_id=999,
                amount=10.0
            )
        )

        insufficient_balance_result = (
            security_test_system.transfer_money(
                sender_id=1,
                receiver_id=2,
                amount=1000.0
            )
        )

        payment_request_result = (
            security_test_system.request_payment(
                requester_id=1,
                payer_id=2,
                amount=20.0
            )
        )

        first_approval_result = (
            security_test_system.approve_payment_request(
                request_id=1
            )
        )

        second_approval_result = (
            security_test_system.approve_payment_request(
                request_id=1
            )
        )

        return {
            "attribute_checks": attribute_checks,
            "method_checks": method_checks,
            "audit_log_exists": isinstance(
                payment_system.audit_log,
                list
            ),
            "audit_log_entries": len(
                payment_system.audit_log
            ),
            "business_rule_checks": {
                "negative_amount_blocked": (
                    negative_amount_result
                    .output["success"]
                    is False
                ),
                "self_transfer_blocked": (
                    self_transfer_result
                    .output["success"]
                    is False
                ),
                "missing_user_blocked": (
                    missing_user_result
                    .output["success"]
                    is False
                ),
                "insufficient_balance_blocked": (
                    insufficient_balance_result
                    .output["success"]
                    is False
                ),
                "first_approval_succeeded": (
                    first_approval_result
                    .output["success"]
                    is True
                ),
                "duplicate_approval_blocked": (
                    second_approval_result
                    .output["success"]
                    is False
                ),
                "payment_request_created": (
                    payment_request_result
                    .output["success"]
                    is True
                )
            },
            "transaction_history_enabled": (
                hasattr(
                    payment_system,
                    "transactions"
                )
                and isinstance(
                    payment_system.transactions,
                    list
                )
            )
        }

time: 5.28 ms (started: 2026-09-28 13:00:20 +00:00)


## SecurityAgent Structured Output

The `SecurityAgent` returns a structured security assessment containing:

- Whether the security review passed.
- A concise summary of the assessment.
- Any security or business-rule issues identified.
- Recommendations for improving the system's security and reliability.

In [112]:
class SecurityReviewOutput(BaseModel):
    passed: bool = Field(
        description="Whether the security review passed."
    )

    summary: str = Field(
        description="A short summary of the security review."
    )

    issues: List[str] = Field(
        default_factory=list,
        description="Security or business-logic issues found."
    )

    recommendations: List[str] = Field(
        default_factory=list,
        description="Recommended improvements."
    )

time: 4.47 ms (started: 2026-09-28 13:00:28 +00:00)


## SecurityAgent Instructions

The `SecurityAgent` evaluates only the security report generated by the deterministic Python inspection tool.

It must base its assessment exclusively on the checks and findings contained in the provided report and must not fabricate additional security tests, vulnerabilities, or system behavior.

The agent performs analysis only and does not modify the `PaymentSystem` or its state.

In [113]:
SECURITY_SYSTEM_PROMPT = """
You are the SecurityAgent of an educational payment system.

Analyze the supplied security inspection report.

Your responsibilities are:

1. Decide whether the required security and business-logic checks passed.
2. Summarize the findings.
3. List any failed checks as issues.
4. Provide recommendations only for failed or missing checks.

Rules:

- Use only the supplied report.
- Do not invent tests.
- Do not modify the payment system.
- The review passes only when:
  - all required attributes exist;
  - all required methods exist;
  - the audit log exists;
  - transaction history is enabled;
  - all supplied business-rule checks are true.
- If every check passes, return an empty issues list.
"""

time: 667 µs (started: 2026-09-28 13:00:32 +00:00)


## SecurityAgent Implementation

The `SecurityAgent` uses the configured OpenAI provider to perform a structured security assessment.

It first invokes the deterministic `SecurityInspectionTool` to generate a security report based on the current `PaymentSystem`. The resulting report is then analyzed by the LLM to identify potential issues, summarize the findings, and provide recommendations.

This design separates deterministic security inspection from LLM-based analysis and explanation.


In [114]:
class SecurityAgent:
    def __init__(
        self,
        provider: LLMProvider,
        inspection_tool: SecurityInspectionTool
    ):
        self.provider = provider
        self.inspection_tool = inspection_tool

    def run(
        self,
        payment_system: PaymentSystem
    ) -> AgentResult:
        inspection_report = (
            self.inspection_tool.inspect(
                payment_system
            )
        )

        result = self.provider.generate_json(
            system_prompt=SECURITY_SYSTEM_PROMPT,
            user_prompt=json.dumps(
                inspection_report,
                ensure_ascii=False,
                indent=2,
                default=str
            ),
            output_model=SecurityReviewOutput
        )

        return AgentResult(
            agent_name="SecurityAgent",
            output=result.model_dump(),
            confidence=0.95,
            metadata={
                "provider": self.provider.provider_name,
                "model": self.provider.model_name,
                "status": "success",
                "inspection_report": inspection_report
            }
        )

time: 1.18 ms (started: 2026-09-28 13:00:35 +00:00)


## SecurityAgent Initialization

The deterministic security inspection tool and the `SecurityAgent` are initialized for use in security analysis workflows.

In [115]:
security_inspection_tool = SecurityInspectionTool()

security_agent = SecurityAgent(
    provider=openai_provider,
    inspection_tool=security_inspection_tool
)

time: 735 µs (started: 2026-09-28 13:00:38 +00:00)


## SecurityAgent Validation

This test evaluates the `SecurityAgent` using a correctly configured payment system.

The deterministic inspection tool is expected to generate a report confirming that the required system structures, payment operations, audit mechanisms, and business-rule safeguards are in place.

The `SecurityAgent` then analyzes this report and returns a structured security assessment.

In [116]:
if RUN_COMPONENT_TESTS:
  security_test_payment_system = PaymentSystem()

  security_test_payment_system.create_user(
      name="Alice",
      phone_number="0501234567",
      initial_balance=500.0
  )

  security_test_payment_system.create_user(
      name="Bob",
      phone_number="0507654321",
      initial_balance=300.0
  )

  security_result = security_agent.run(
      security_test_payment_system
  )

  security_result.display()

Agent name: SecurityAgent
Confidence: 0.95
Output:
{
  "passed": true,
  "summary": "All required attributes and methods are present. Audit log exists with entries, transaction history is enabled, and all supplied business-rule checks passed.",
  "issues": [],
  "recommendations": []
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "success",
  "inspection_report": {
    "attribute_checks": {
      "users": true,
      "wallets": true,
      "transactions": true,
      "payment_requests": true,
      "audit_log": true
    },
    "method_checks": {
      "create_user": true,
      "get_balance": true,
      "transfer_money": true,
      "get_transactions": true,
      "request_payment": true,
      "approve_payment_request": true,
      "reject_payment_request": true
    },
    "audit_log_exists": true,
    "audit_log_entries": 2,
    "business_rule_checks": {
      "negative_amount_blocked": true,
      "self_transfer_blocked": true,
      "missing_user_blocke

In [117]:
if RUN_COMPONENT_TESTS:
  inspection_report = (
      security_result.metadata[
          "inspection_report"
      ]
  )

  assert all(
      inspection_report[
          "attribute_checks"
      ].values()
  )

  assert all(
      inspection_report[
          "method_checks"
      ].values()
  )

  assert inspection_report[
      "audit_log_exists"
  ] is True

  assert inspection_report[
      "transaction_history_enabled"
  ] is True

  assert all(
      inspection_report[
          "business_rule_checks"
      ].values()
  )

  assert security_result.output["passed"] is True

  print("SecurityAgent review test passed.")

SecurityAgent review test passed.
time: 1.49 ms (started: 2026-09-28 13:00:57 +00:00)


# Critic Agent

The `CriticAgent` evaluates the quality and consistency of results produced by other components in the system.

The agent receives:

- The original user request.
- The detected intent.
- The name of the component that handled the operation.
- The result produced by that component.

It evaluates whether the result is clear, consistent, and aligned with the user's request.

If the result does not meet the expected quality criteria, the `CriticAgent` provides structured feedback and determines whether the request should be redirected to the fallback mechanism.

## CriticAgent Structured Output

The `CriticAgent` returns a structured quality assessment containing:

- A quality score from 1 to 5.
- Whether the evaluated result is approved.
- Whether the `FallbackAgent` should be triggered.
- The reasons supporting the evaluation.
- Feedback for improving the result.

In [118]:
class CriticOutput(BaseModel):
    quality_score: int = Field(
        ge=1,
        le=5,
        description="The quality score of the reviewed result."
    )

    approved: bool = Field(
        description="Whether the result is acceptable."
    )

    requires_fallback: bool = Field(
        description="Whether FallbackAgent should be activated."
    )

    reasons: List[str] = Field(
        default_factory=list,
        description="Reasons supporting the evaluation."
    )

    feedback: str = Field(
        description="Short feedback for improving the result."
    )

time: 5.06 ms (started: 2026-09-28 13:01:04 +00:00)


## CriticAgent Instructions

The `CriticAgent` evaluates only the quality of the result it receives and does not re-execute the original operation or modify the `PaymentSystem`.

Its assessment focuses on whether the result is clear, consistent, and appropriate for the user's request.

A rejected business operation is not automatically considered a low-quality result. If the rejection correctly enforces the system's business rules and clearly communicates the outcome, the `CriticAgent` may still approve it as a valid and high-quality response.

In [119]:
CRITIC_SYSTEM_PROMPT = """
You are the CriticAgent of an educational digital payment system.

Evaluate the supplied operation result.

Your responsibilities are:

1. Check whether the result matches the detected intent.
2. Check whether the output is clear and structured.
3. Check whether success or rejection is explained.
4. Return a quality score from 1 to 5.
5. Decide whether the result is acceptable.
6. Decide whether FallbackAgent is required.

Evaluation rules:

- A successful and complete business result usually receives 5.
- A correctly rejected operation with a clear reason may receive 4 or 5.
- A valid but incomplete explanation may receive 3.
- A confusing, empty or unusable result receives 1 or 2.
- Do not reject a result only because the business operation failed.
- A valid business-rule rejection is still a correct system result.
- requires_fallback must be true only when the result is unclear,
  unusable, incomplete or unrelated to the user's request.
- Use only the supplied data.
- Do not invent missing outputs.
"""

time: 738 µs (started: 2026-09-28 13:01:11 +00:00)


## CriticAgent Implementation

The `CriticAgent` uses the configured OpenAI provider to evaluate the quality of results produced by other system components.

It receives structured context describing the original request, detected intent, responsible component, and operation result. The agent then returns its evaluation using the standardized `AgentResult` format.

This allows the critic's assessment to integrate consistently with the rest of the multi-agent workflow.

In [120]:
class CriticAgent:
    def __init__(
        self,
        provider: LLMProvider
    ):
        self.provider = provider

    def run(
        self,
        user_message: str,
        intent: str,
        result: AgentResult
    ) -> AgentResult:
        critic_context = {
            "user_message": user_message,
            "detected_intent": intent,
            "reviewed_agent": result.agent_name,
            "result_output": result.output,
            "result_metadata": result.metadata
        }

        critique = self.provider.generate_json(
            system_prompt=CRITIC_SYSTEM_PROMPT,
            user_prompt=json.dumps(
                critic_context,
                ensure_ascii=False,
                indent=2,
                default=str
            ),
            output_model=CriticOutput
        )

        return AgentResult(
            agent_name="CriticAgent",
            output=critique.model_dump(),
            confidence=0.95,
            metadata={
                "provider": self.provider.provider_name,
                "model": self.provider.model_name,
                "status": "success",
                "reviewed_agent": result.agent_name
            }
        )

time: 1.89 ms (started: 2026-09-28 13:01:15 +00:00)


## CriticAgent Initialization

The `CriticAgent` is initialized with the configured OpenAI provider.

In [121]:
critic_agent = CriticAgent(
    provider=openai_provider
)

time: 388 µs (started: 2026-09-28 13:01:18 +00:00)




## Tests

### CriticAgent Successful Operation Test

This test evaluates the `CriticAgent` using the result of a successful money transfer.

The agent is expected to recognize that the result is clear, structured, and consistent with the `transferMoney` intent.

In [122]:
if RUN_COMPONENT_TESTS:
  critic_test_system = PaymentSystem()

  critic_test_system.create_user(
      name="Alice",
      phone_number="0501234567",
      initial_balance=500.0
  )

  critic_test_system.create_user(
      name="Bob",
      phone_number="0507654321",
      initial_balance=300.0
  )

  critic_success_result = critic_test_system.transfer_money(
      sender_id=1,
      receiver_id=2,
      amount=100.0
  )

  successful_critique = critic_agent.run(
      user_message="Transfer 100 from user 1 to user 2",
      intent=Intent.TRANSFER_MONEY.value,
      result=critic_success_result
  )

  successful_critique.display()

Agent name: CriticAgent
Confidence: 0.95
Output:
{
  "quality_score": 5,
  "approved": true,
  "requires_fallback": false,
  "reasons": [
    "Output matches the detected intent (transferMoney) and indicates a successful transfer.",
    "Result is clear and well-structured JSON with a success flag, human-readable message, and detailed transaction object (id, sender/receiver ids, amount, timestamp, status, risk_score).",
    "Balances are provided and consistent with metadata (sender_balance_before 500.0 -> sender_balance 400.0 after a 100.0 transfer).",
    "Result metadata aligns with the main result (action, status, transaction_id).",
    "Success is explicitly explained by the message and supported by transaction status and balances."
  ],
  "feedback": "Result is complete and correct. For extra clarity/auditability consider including receiver_balance_before, currency, any fees, and a confirmation/reference code in future responses."
}
Metadata:
{
  "provider": "OpenAI",
  "model": 

In [123]:
if RUN_COMPONENT_TESTS:
  assert successful_critique.agent_name == "CriticAgent"

  assert (
      successful_critique.output["approved"]
      is True
  )

  assert (
      successful_critique.output["requires_fallback"]
      is False
  )

  assert (
      successful_critique.output["quality_score"]
      >= 3
  )

  print("CriticAgent successful-result test passed.")

CriticAgent successful-result test passed.
time: 808 µs (started: 2026-09-28 13:01:41 +00:00)


### CriticAgent Rejected Operation Test

This test evaluates the `CriticAgent` using an operation that was rejected due to insufficient funds.

The agent is expected to recognize that the rejection is a valid business-rule outcome rather than a faulty result, provided that the reason for the rejection is clear and consistent with the system's logic.

In [124]:
if RUN_COMPONENT_TESTS:
  critic_rejected_result = critic_test_system.transfer_money(
      sender_id=1,
      receiver_id=2,
      amount=5000.0
  )

  rejected_critique = critic_agent.run(
      user_message="Transfer 5000 from user 1 to user 2",
      intent=Intent.TRANSFER_MONEY.value,
      result=critic_rejected_result
  )

  rejected_critique.display()

Agent name: CriticAgent
Confidence: 0.95
Output:
{
  "quality_score": 5,
  "approved": true,
  "requires_fallback": false,
  "reasons": [
    "Output matches the detected intent (transferMoney) and the reviewed agent action (transfer_money).",
    "The response is structured and clearly indicates the operation was rejected (success: false).",
    "A concise reason for rejection is provided: \"Insufficient balance.\"",
    "Result metadata status (rejected) is consistent with the success flag and message."
  ],
  "feedback": "Result is valid and clear. For improved user experience, include additional context such as the current available balance, a suggested next step (e.g., top-up or try a smaller amount), and optionally a transaction reference or error code for support."
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "success",
  "reviewed_agent": "PaymentSystem"
}
time: 8.03 s (started: 2026-09-28 13:01:44 +00:00)


In [125]:
if RUN_COMPONENT_TESTS:
  assert (
      rejected_critique.output["approved"]
      is True
  )

  assert (
      rejected_critique.output["requires_fallback"]
      is False
  )

  assert (
      rejected_critique.output["quality_score"]
      >= 3
  )

  print("CriticAgent rejected-result test passed.")

CriticAgent rejected-result test passed.
time: 1.21 ms (started: 2026-09-28 13:01:55 +00:00)


# Reflection Agent

The `ReflectionAgent` provides structured feedback after a business operation is rejected.

Its responsibilities include:

- Explaining why the operation failed.
- Suggesting a valid way to correct or reformulate the request.
- Ensuring that suggested alternatives remain consistent with the payment system's business rules.
- Preventing recommendations that would bypass system constraints.

The `ReflectionAgent` does not modify the failed operation or automatically retry it. Its role is limited to analyzing the failure and providing guidance for a valid next step.

## ReflectionAgent Structured Output

The `ReflectionAgent` returns a structured response containing:

- A summary of the error or failed operation.
- A suggested correction or alternative approach.
- The fields or parameters that should be modified.
- Whether the request can be retried after correction.

In [133]:
class ReflectionOutput(BaseModel):
    error_summary: str = Field(
        description="A short summary of the business error."
    )

    correction_suggestion: str = Field(
        description="A safe suggestion for correcting the request."
    )

    fields_to_review: List[str] = Field(
        default_factory=list,
        description="Fields that the user should review."
    )

    can_retry: bool = Field(
        description="Whether the user can retry with corrected data."
    )

time: 4.52 ms (started: 2026-09-28 13:17:28 +00:00)


## ReflectionAgent Instructions

The `ReflectionAgent` bases its analysis only on the original user request and the error returned by the failed operation.

It must not suggest ways to bypass business rules, fabricate users or identifiers, modify wallet balances, or introduce information that was not provided by the system.

Any suggested correction must remain consistent with the existing payment-system constraints.

In [134]:
REFLECTION_SYSTEM_PROMPT = """
You are the ReflectionAgent of an educational digital payment system.

Analyze a rejected business operation and suggest a safe correction.

Your responsibilities are:

1. Summarize the error.
2. Explain what the user should correct.
3. Identify which input fields should be reviewed.
4. Decide whether the user can retry.

Rules:

- Do not bypass business rules.
- Do not invent user identifiers.
- Do not invent payment-request identifiers.
- Do not invent balances or amounts.
- Do not claim that the failed operation succeeded.
- If the error is insufficient balance, suggest a lower amount
  or checking the balance.
- If a user does not exist, suggest checking the user identifier.
- If the amount is invalid, explain that it must be greater than zero.
- If the transfer is self-directed, require different user identifiers.
- If a request was already approved or rejected, explain that only
  pending requests can be processed.
- Keep the answer short and practical.
"""

time: 1.25 ms (started: 2026-09-28 13:17:31 +00:00)


## ReflectionAgent Implementation

The `ReflectionAgent` uses the configured OpenAI provider to analyze rejected business operations.

It receives the original user request together with the result of the failed operation and generates structured feedback describing the failure and a valid path for correction.

The response is returned using the standardized `AgentResult` format, allowing it to integrate consistently with the rest of the multi-agent workflow.


In [135]:
class ReflectionAgent:
    def __init__(
        self,
        provider: LLMProvider
    ):
        self.provider = provider

    def run(
        self,
        user_message: str,
        result: AgentResult
    ) -> AgentResult:
        reflection_context = {
            "user_message": user_message,
            "reviewed_agent": result.agent_name,
            "result_output": result.output,
            "result_metadata": result.metadata
        }

        reflection = self.provider.generate_json(
            system_prompt=REFLECTION_SYSTEM_PROMPT,
            user_prompt=json.dumps(
                reflection_context,
                ensure_ascii=False,
                indent=2,
                default=str
            ),
            output_model=ReflectionOutput
        )

        return AgentResult(
            agent_name="ReflectionAgent",
            output=reflection.model_dump(),
            confidence=0.95,
            metadata={
                "provider": self.provider.provider_name,
                "model": self.provider.model_name,
                "status": "success",
                "reviewed_agent": result.agent_name
            }
        )

time: 1.5 ms (started: 2026-09-28 13:17:35 +00:00)


## ReflectionAgent Initialization

The `ReflectionAgent` is initialized with the configured OpenAI provider.

In [136]:
reflection_agent = ReflectionAgent(
    provider=openai_provider
)

time: 460 µs (started: 2026-09-28 13:17:39 +00:00)


## ReflectionAgent Validation

This test evaluates the `ReflectionAgent` using a money transfer that was rejected due to insufficient funds.

The agent is expected to explain the reason for the rejection and suggest a valid corrective action, such as checking the available balance or retrying the transfer with a lower amount.

In [137]:
if RUN_COMPONENT_TESTS:
  reflection_test_result = reflection_agent.run(
      user_message="Transfer 5000 from user 1 to user 2",
      result=critic_rejected_result
  )

  reflection_test_result.display()

Agent name: ReflectionAgent
Confidence: 0.95
Output:
{
  "error_summary": "Transfer rejected due to insufficient balance for the requested transfer of 5000 from user 1 to user 2.",
  "correction_suggestion": "Check user 1's current account balance and either top up the account or reduce the transfer amount to one within the available balance. Alternatively choose a different funding source for the payment and then retry.",
  "fields_to_review": [
    "sender_id (user 1)",
    "recipient_id (user 2)",
    "amount (5000)",
    "sender account balance / funding source"
  ],
  "can_retry": true
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "success",
  "reviewed_agent": "PaymentSystem"
}
time: 5.98 s (started: 2026-09-28 13:17:41 +00:00)


In [138]:
if RUN_COMPONENT_TESTS:
  assert (
      reflection_test_result.agent_name
      == "ReflectionAgent"
  )

  assert (
      reflection_test_result.output["can_retry"]
      is True
  )

  assert (
      len(
          reflection_test_result.output[
              "correction_suggestion"
          ]
      )
      > 0
  )

  assert (
      len(
          reflection_test_result.output[
              "fields_to_review"
          ]
      )
      > 0
  )

  print("ReflectionAgent test passed.")

ReflectionAgent test passed.
time: 918 µs (started: 2026-09-28 13:19:03 +00:00)


# Orchestrator Agent

The `OrchestratorAgent` manages the system workflow after the `RouterAgent` identifies the user's intent.

Its responsibility is to determine which component should handle the request:

- **PaymentSystem** for payment operations.
- **FraudDetectionAgent** for fraud-risk assessment.
- **SecurityAgent** for security reviews.
- **ExplanationAgent** for explanations of recent operations.
- **FallbackAgent** for unclear or incomplete requests.

The `OrchestratorAgent` does not modify wallet balances or execute Python tools directly. Instead, it determines the appropriate workflow, while the `OrchestratorRuntime` is responsible for executing the selected action.

In [139]:
class ExecutionTarget(str, Enum):
    PAYMENT_SYSTEM = "payment_system"
    FRAUD_DETECTION_AGENT = "fraud_detection_agent"
    SECURITY_AGENT = "security_agent"
    EXPLANATION_AGENT = "explanation_agent"
    FALLBACK_AGENT = "fallback_agent"

time: 1.26 ms (started: 2026-09-28 13:19:07 +00:00)


## OrchestratorAgent Structured Output

The `OrchestratorAgent` returns a structured orchestration decision containing:

- The component responsible for handling the request.
- The action to be performed.
- The reasoning behind the routing decision.
- Whether the shared business memory should be updated.
- Whether the `CriticAgent` should evaluate the result after execution.

Business parameters are taken directly from the structured output produced by the `RouterAgent`. The `OrchestratorAgent` does not recreate, infer, or modify these parameters.

In [140]:
class OrchestratorDecision(BaseModel):
    selected_component: ExecutionTarget = Field(
        description="The component selected to handle the request."
    )

    action: str = Field(
        description="The business or agent action to execute."
    )

    reason: str = Field(
        description="A short explanation for selecting the component."
    )

    update_memory: bool = Field(
        description="Whether business memory should be updated."
    )

    run_critic: bool = Field(
        description="Whether CriticAgent should review the result."
    )

time: 6.11 ms (started: 2026-09-28 13:19:09 +00:00)


## OrchestratorAgent Instructions

The `OrchestratorAgent` receives the structured output from the `RouterAgent` together with the current state of the shared business memory.

It must select exactly one appropriate component to handle the request based on the detected intent and available context.

The agent must not fabricate parameters, users, transactions, or unsupported operations. Its responsibility is limited to selecting and coordinating the appropriate workflow using information already available within the system.


In [141]:
ORCHESTRATOR_SYSTEM_PROMPT = """
You are the OrchestratorAgent of an educational digital payment system.

You receive:

1. A structured RouterAgent result.
2. A summary of the business memory.

Choose exactly one system component.

Routing rules:

- createUser -> payment_system, action=create_user
- checkBalance -> payment_system, action=get_balance
- transferMoney -> payment_system, action=transfer_money
- requestPayment -> payment_system, action=request_payment
- approvePayment -> payment_system, action=approve_payment_request
- rejectPayment -> payment_system, action=reject_payment_request
- showTransactions -> payment_system, action=get_transactions
- fraudCheck -> fraud_detection_agent, action=fraud_check
- securityReview -> security_agent, action=security_review
- explainLastAction -> explanation_agent, action=explain_last_action
- unknown -> fallback_agent, action=fallback

Rules:

- Do not invent parameters.
- Do not convert names into user identifiers.
- Do not perform financial operations.
- Select fallback_agent for unknown intent.
- Payment operations must be executed only by payment_system.
- Fraud analysis must be executed only by fraud_detection_agent.
- Security review must be executed only by security_agent.
- Explanation must be executed only by explanation_agent.
- Set run_critic to true for every non-fallback result.
- Set update_memory to true for every executed request.
"""

time: 1.11 ms (started: 2026-09-28 13:19:13 +00:00)


## OrchestratorAgent Implementation

The `OrchestratorAgent` uses the configured OpenAI provider to coordinate the multi-agent workflow.

It receives the structured routing result from the `RouterAgent` together with the current `BusinessMemory` context and returns a structured orchestration decision.

The resulting decision specifies which component should handle the request and how the workflow should proceed, while the actual execution remains the responsibility of the `OrchestratorRuntime`.

In [142]:
class OrchestratorAgent:
    def __init__(
        self,
        provider: LLMProvider
    ):
        self.provider = provider

    def run(
        self,
        route_result: AgentResult,
        memory: BusinessMemory
    ) -> AgentResult:
        orchestrator_context = {
            "router_result": route_result.output,
            "memory_summary": {
                "last_intent": memory.last_intent,
                "last_action": memory.last_action,
                "has_last_transaction": (
                    memory.last_transaction is not None
                ),
                "has_last_payment_request": (
                    memory.last_payment_request is not None
                )
            }
        }

        decision = self.provider.generate_json(
            system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
            user_prompt=json.dumps(
                orchestrator_context,
                ensure_ascii=False,
                indent=2,
                default=str
            ),
            output_model=OrchestratorDecision
        )

        return AgentResult(
            agent_name="OrchestratorAgent",
            output=decision.model_dump(),
            confidence=0.95,
            metadata={
                "provider": self.provider.provider_name,
                "model": self.provider.model_name,
                "status": "success",
                "router_intent": route_result.output["intent"]
            }
        )

time: 1.37 ms (started: 2026-09-28 13:19:18 +00:00)


## OrchestratorAgent Initialization

The `OrchestratorAgent` is initialized with the configured OpenAI provider.

In [143]:
orchestrator_agent = OrchestratorAgent(
    provider=openai_provider
)

time: 609 µs (started: 2026-09-28 13:19:22 +00:00)


## OrchestratorAgent Decision Test

This test first processes a money-transfer request through the `RouterAgent` and then passes the structured routing result to the `OrchestratorAgent`.

The `OrchestratorAgent` is expected to select the `PaymentSystem` as the responsible component and `transfer_money` as the appropriate action.

In [144]:
if RUN_COMPONENT_TESTS:
  orchestrator_route_test = router_agent.run(
      "Transfer 100 from user 1 to user 2"
  )

  orchestrator_decision_test = orchestrator_agent.run(
      route_result=orchestrator_route_test,
      memory=business_memory
  )

  orchestrator_decision_test.display()

Agent name: OrchestratorAgent
Confidence: 0.95
Output:
{
  "selected_component": "payment_system",
  "action": "transfer_money",
  "reason": "Intent 'transferMoney' requires routing to payment_system to perform the transfer between the provided sender and receiver.",
  "update_memory": true,
  "run_critic": true
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "success",
  "router_intent": "transferMoney"
}
time: 8.06 s (started: 2026-09-28 13:19:26 +00:00)


In [145]:
if RUN_COMPONENT_TESTS:
  assert (
      orchestrator_decision_test.output[
          "selected_component"
      ]
      == ExecutionTarget.PAYMENT_SYSTEM.value
  )

  assert (
      orchestrator_decision_test.output["action"]
      == "transfer_money"
  )

  assert (
      orchestrator_decision_test.output["run_critic"]
      is True
  )

  print("OrchestratorAgent decision test passed.")

OrchestratorAgent decision test passed.
time: 767 µs (started: 2026-09-28 13:19:37 +00:00)


# Orchestrator Runtime

The `OrchestratorRuntime` connects the decisions produced by the LLM-based agents with the deterministic execution layer of the system.

Its responsibilities include:

1. Receiving the user's request.
2. Invoking the `RouterAgent`.
3. Invoking the `OrchestratorAgent`.
4. Validating that all required parameters are available.
5. Executing the appropriate `PaymentSystem` operation or specialized agent.
6. Updating the shared `BusinessMemory`.
7. Invoking the `FraudDetectionAgent` after a successful money transfer.
8. Invoking the `SecurityAgent` when a transaction is identified as suspicious.
9. Invoking the `ReflectionAgent` after a rejected business operation.
10. Invoking the `CriticAgent` to evaluate the resulting output.
11. Invoking the `FallbackAgent` when the request cannot be processed reliably.

The `OrchestratorRuntime` is a deterministic Python component rather than an LLM agent. It does not make natural-language decisions. Instead, it executes the structured decisions produced by the agents and coordinates the overall workflow.

In [146]:
class OrchestratorRuntime:
    def __init__(
        self,
        router_agent: RouterAgent,
        orchestrator_agent: OrchestratorAgent,
        payment_system: PaymentSystem,
        memory: BusinessMemory,
        fallback_agent: FallbackAgent,
        fraud_detection_agent: FraudDetectionAgent,
        security_agent: SecurityAgent,
        explanation_agent: ExplanationAgent,
        critic_agent: CriticAgent,
        reflection_agent: ReflectionAgent
    ):
        self.router_agent = router_agent
        self.orchestrator_agent = orchestrator_agent
        self.payment_system = payment_system
        self.memory = memory
        self.fallback_agent = fallback_agent
        self.fraud_detection_agent = fraud_detection_agent
        self.security_agent = security_agent
        self.explanation_agent = explanation_agent
        self.critic_agent = critic_agent
        self.reflection_agent = reflection_agent

    def _required_fields(
        self,
        intent: str
    ) -> List[str]:
        fields_by_intent = {
            Intent.CREATE_USER.value: [
                "name",
                "phone_number",
                "initial_balance"
            ],
            Intent.CHECK_BALANCE.value: [
                "user_id"
            ],
            Intent.TRANSFER_MONEY.value: [
                "sender_id",
                "receiver_id",
                "amount"
            ],
            Intent.REQUEST_PAYMENT.value: [
                "requester_id",
                "payer_id",
                "amount"
            ],
            Intent.APPROVE_PAYMENT.value: [
                "request_id"
            ],
            Intent.REJECT_PAYMENT.value: [
                "request_id"
            ],
            Intent.SHOW_TRANSACTIONS.value: [
                "user_id"
            ]
        }

        return fields_by_intent.get(
            intent,
            []
        )

    def _find_missing_fields(
        self,
        intent: str,
        parameters: Dict[str, Any]
    ) -> List[str]:
        required_fields = self._required_fields(
            intent
        )

        return [
            field
            for field in required_fields
            if (
                field not in parameters
                or parameters[field] is None
            )
        ]

    def _execute_payment_action(
        self,
        intent: str,
        parameters: Dict[str, Any]
    ) -> AgentResult:
        if intent == Intent.CREATE_USER.value:
            return self.payment_system.create_user(
                name=parameters["name"],
                phone_number=parameters["phone_number"],
                initial_balance=parameters["initial_balance"]
            )

        if intent == Intent.CHECK_BALANCE.value:
            return self.payment_system.get_balance(
                user_id=parameters["user_id"]
            )

        if intent == Intent.TRANSFER_MONEY.value:
            return self.payment_system.transfer_money(
                sender_id=parameters["sender_id"],
                receiver_id=parameters["receiver_id"],
                amount=parameters["amount"]
            )

        if intent == Intent.REQUEST_PAYMENT.value:
            return self.payment_system.request_payment(
                requester_id=parameters["requester_id"],
                payer_id=parameters["payer_id"],
                amount=parameters["amount"]
            )

        if intent == Intent.APPROVE_PAYMENT.value:
            return self.payment_system.approve_payment_request(
                request_id=parameters["request_id"]
            )

        if intent == Intent.REJECT_PAYMENT.value:
            return self.payment_system.reject_payment_request(
                request_id=parameters["request_id"]
            )

        if intent == Intent.SHOW_TRANSACTIONS.value:
            return self.payment_system.get_transactions(
                user_id=parameters["user_id"]
            )

        return self.fallback_agent.run(
            user_message="Unsupported payment action.",
            reason="The payment action is not supported.",
            intent=intent
        )

    def _count_sender_transactions(
        self,
        sender_id: int
    ) -> int:
        return sum(
            1
            for transaction in self.payment_system.transactions
            if transaction.sender_id == sender_id
        )

    def _store_transaction_risk_score(
        self,
        transaction: Dict[str, Any],
        risk_score: float
    ) -> None:
        transaction_id = transaction["transaction_id"]

        transaction["risk_score"] = float(risk_score)

        for stored_transaction in self.payment_system.transactions:
            if stored_transaction.transaction_id == transaction_id:
                stored_transaction.risk_score = float(risk_score)
                break

    def _update_memory(
        self,
        user_message: str,
        intent: str,
        selected_component: str,
        result: AgentResult
    ) -> None:
        user = None
        transaction = None
        payment_request = None

        if isinstance(result.output, dict):
            user = result.output.get("user")
            transaction = result.output.get(
                "transaction"
            )
            payment_request = result.output.get(
                "payment_request"
            )

        if user is None:
            user_id = None

            if isinstance(result.metadata, dict):
                user_id = result.metadata.get(
                    "user_id"
                )

            if (
                user_id is not None
                and user_id in self.payment_system.users
            ):
                user = asdict(
                    self.payment_system.users[user_id]
                )

        if user is None and transaction is not None:
            sender_id = transaction.get("sender_id")

            if (
                sender_id is not None
                and sender_id in self.payment_system.users
            ):
                user = asdict(
                    self.payment_system.users[sender_id]
                )

        if user is None and payment_request is not None:
            requester_id = payment_request.get(
                "requester_id"
            )

            if (
                requester_id is not None
                and requester_id in self.payment_system.users
            ):
                user = asdict(
                    self.payment_system.users[requester_id]
                )

        action = intent

        if isinstance(result.metadata, dict):
            action = result.metadata.get(
                "action",
                intent
            )

        update_business_memory(
            memory=self.memory,
            intent=intent,
            tool=selected_component,
            user_message=user_message,
            action=action,
            user=user,
            transaction=transaction,
            payment_request=payment_request,
            result=(
                result.output
                if isinstance(result.output, dict)
                else {"output": result.output}
            )
        )

    def run(
        self,
        user_message: str
    ) -> AgentResult:
        route_result = self.router_agent.run(
            user_message
        )

        intent = route_result.output["intent"]
        parameters = route_result.output.get(
            "parameters",
            {}
        )
        router_confidence = route_result.confidence

        if (
            intent == Intent.UNKNOWN.value
            or router_confidence < 0.80
        ):
            fallback_result = self.fallback_agent.run(
                user_message=user_message,
                reason=(
                    "The intent is unknown or the routing "
                    "confidence is too low."
                ),
                intent=intent
            )

            self._update_memory(
                user_message=user_message,
                intent=intent,
                selected_component="fallback_agent",
                result=fallback_result
            )

            fallback_result.metadata = {
                **(fallback_result.metadata or {}),
                "router_result": route_result.output,
                "orchestrator_decision": None,
                "critic_result": None,
                "fraud_result": None,
                "security_result": None,
                "reflection_result": None
            }
            return fallback_result

        decision_result = self.orchestrator_agent.run(
            route_result=route_result,
            memory=self.memory
        )

        selected_component = (
            decision_result.output[
                "selected_component"
            ]
        )

        if selected_component == "fallback_agent":
            result = self.fallback_agent.run(
                user_message=user_message,
                reason=decision_result.output["reason"],
                intent=intent
            )

            self._update_memory(
                user_message=user_message,
                intent=intent,
                selected_component=selected_component,
                result=result
            )
            result.metadata = {
                **(result.metadata or {}),
                "router_result": route_result.output,
                "orchestrator_decision": decision_result.output,
                "critic_result": None,
                "fraud_result": None,
                "security_result": None,
                "reflection_result": None
            }

            return result

        missing_fields = self._find_missing_fields(
            intent=intent,
            parameters=parameters
        )

        if missing_fields:
            result = self.fallback_agent.run(
                user_message=user_message,
                reason="Required action parameters are missing.",
                intent=intent,
                missing_fields=missing_fields
            )

            self._update_memory(
                user_message=user_message,
                intent=intent,
                selected_component="fallback_agent",
                result=result
            )
            result.metadata = {
                **(result.metadata or {}),
                "router_result": route_result.output,
                "orchestrator_decision": decision_result.output,
                "critic_result": None,
                "fraud_result": None,
                "security_result": None,
                "reflection_result": None
            }

            return result

        if selected_component == "payment_system":
            result = self._execute_payment_action(
                intent=intent,
                parameters=parameters
            )

        elif selected_component == "explanation_agent":
            result = self.explanation_agent.run(
                self.memory
            )

        elif selected_component == "security_agent":
            result = self.security_agent.run(
                self.payment_system
            )

        elif selected_component == "fraud_detection_agent":
            last_transaction = (
                self.memory.last_transaction
            )

            if last_transaction is None:
                result = self.fallback_agent.run(
                    user_message=user_message,
                    reason=(
                        "There is no previous transaction "
                        "available for fraud analysis."
                    ),
                    intent=intent
                )

                self._update_memory(
                    user_message=user_message,
                    intent=intent,
                    selected_component="fallback_agent",
                    result=result
                )

                result.metadata = {
                    **(result.metadata or {}),
                    "router_result": route_result.output,
                    "orchestrator_decision": decision_result.output,
                    "critic_result": None,
                    "fraud_result": None,
                    "security_result": None,
                    "reflection_result": None
                }
                return result

            sender_id = last_transaction["sender_id"]

            sender_balance_after = (
                self.payment_system
                .wallets[sender_id]
                .balance
            )

            sender_balance_before = (
                sender_balance_after
                + last_transaction["amount"]
            )

            result = self.fraud_detection_agent.run(
                transaction=last_transaction,
                sender_balance_before=sender_balance_before,
                sender_transaction_count=(
                    self._count_sender_transactions(
                        sender_id
                    )
                )
            )
            self._store_transaction_risk_score(
                transaction=last_transaction,
                risk_score=result.output["risk_score"]
            )

        else:
            result = self.fallback_agent.run(
                user_message=user_message,
                reason="No valid component was selected.",
                intent=intent
            )

        if intent in [
            Intent.EXPLAIN_LAST_ACTION.value,
            Intent.FRAUD_CHECK.value,
            Intent.SECURITY_REVIEW.value
        ]:
            update_business_memory(
                memory=self.memory,
                intent=intent,
                tool=selected_component,
                user_message=user_message
            )

        else:
            self._update_memory(
                user_message=user_message,
                intent=intent,
                selected_component=selected_component,
                result=result
            )

        fraud_result = None
        security_result = None
        reflection_result = None

        if (
            intent in [
                Intent.TRANSFER_MONEY.value,
                Intent.APPROVE_PAYMENT.value
            ]
            and isinstance(result.output, dict)
            and result.output.get("success") is True
            and "transaction" in result.output
        ):
            transaction = result.output["transaction"]

            sender_id = transaction["sender_id"]

            sender_balance_after = (
                self.payment_system
                .wallets[sender_id]
                .balance
            )

            sender_balance_before = (
                sender_balance_after
                + transaction["amount"]
            )

            fraud_result = self.fraud_detection_agent.run(
                transaction=transaction,
                sender_balance_before=sender_balance_before,
                sender_transaction_count=(
                    self._count_sender_transactions(
                        sender_id
                    )
                )
            )

            self._store_transaction_risk_score(
                transaction=transaction,
                risk_score=fraud_result.output["risk_score"]
            )

            if fraud_result.output["suspicious"] is True:
                security_result = self.security_agent.run(
                    self.payment_system
                )


        if (
            isinstance(result.output, dict)
            and result.output.get("success") is False
        ):
            reflection_result = self.reflection_agent.run(
                user_message=user_message,
                result=result
            )

        critique_result = self.critic_agent.run(
            user_message=user_message,
            intent=intent,
            result=result
        )

        update_business_memory(
            memory=self.memory,
            critic_score=(
                critique_result.output[
                    "quality_score"
                ]
            )
        )

        if (
            critique_result.output[
                "requires_fallback"
            ]
            is True
        ):
            fallback_result = self.fallback_agent.run(
                user_message=user_message,
                reason=(
                    "CriticAgent determined that the "
                    "result was not usable."
                ),
                intent=intent
            )

            return fallback_result

        combined_metadata = dict(
            result.metadata or {}
        )

        combined_metadata.update({
            "router_result": route_result.output,
            "orchestrator_decision": (
                decision_result.output
            ),
            "critic_result": (
                critique_result.output
            ),
            "fraud_result": (
                fraud_result.output
                if fraud_result is not None
                else None
            ),
            "security_result": (
                security_result.output
                if security_result is not None
                else None
            ),
            "reflection_result": (
                reflection_result.output
                if reflection_result is not None
                else None
            )
        })

        return AgentResult(
            agent_name=result.agent_name,
            output=result.output,
            confidence=result.confidence,
            metadata=combined_metadata
        )

time: 11 ms (started: 2026-09-28 13:19:41 +00:00)


In [147]:
def display_agent_execution(
    title: str,
    user_message: str,
    result: AgentResult
) -> None:

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    print("\n USER")
    print("-" * 100)
    print(user_message)

    metadata = result.metadata or {}

    def show_section(
        section_title: str,
        data: Any
    ) -> None:
        print(f"\n{section_title}")
        print("-" * 100)

        if data is None:
            print("Not Executed")
            return

        print(json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
            default=str
        ))

    show_section(
        "RouterAgent",
        metadata.get("router_result")
    )

    show_section(
        "OrchestratorAgent",
        metadata.get("orchestrator_decision")
    )

    show_section(
        "FraudDetectionAgent",
        metadata.get("fraud_result")
    )

    show_section(
        "SecurityAgent",
        metadata.get("security_result")
    )

    show_section(
        "ReflectionAgent",
        metadata.get("reflection_result")
    )

    show_section(
        "CriticAgent",
        metadata.get("critic_result")
    )

    print("\nFINAL RESULT")
    print("-" * 100)

    print("Returned by:", result.agent_name)
    print("Confidence:", result.confidence)

    print(json.dumps(
        result.output,
        indent=2,
        ensure_ascii=False,
        default=str
    ))

    print("\n" + "=" * 100)

time: 1.88 ms (started: 2026-09-28 13:19:55 +00:00)


## Integrated System Initialization

A fresh `PaymentSystem`, `BusinessMemory`, and `OrchestratorRuntime` are initialized to create the complete integrated workflow.

The runtime connects the specialized agents, deterministic tools, payment logic, and shared memory into a single coordinated system.

From this point forward, validation scenarios are executed through the orchestration layer rather than by calling the `PaymentSystem` directly, allowing the complete end-to-end workflow to be tested.

In [148]:
integrated_payment_system = PaymentSystem()
integrated_memory = BusinessMemory()

orchestrator_runtime = OrchestratorRuntime(
    router_agent=router_agent,
    orchestrator_agent=orchestrator_agent,
    payment_system=integrated_payment_system,
    memory=integrated_memory,
    fallback_agent=fallback_agent,
    fraud_detection_agent=fraud_detection_agent,
    security_agent=security_agent,
    explanation_agent=explanation_agent,
    critic_agent=critic_agent,
    reflection_agent=reflection_agent
)

print("Integrated agent system was created successfully.")

Integrated agent system was created successfully.
time: 1.12 ms (started: 2026-09-28 13:20:01 +00:00)


<div dir="rtl">

##Tests

</div>

### Integration Test: User Creation

This test submits a natural-language request to create a new user and validates the complete end-to-end workflow.

The request is processed through:

1. `RouterAgent` – Identifies the `createUser` intent and extracts the required parameters.
2. `OrchestratorAgent` – Selects the appropriate component and action.
3. `PaymentSystem` – Executes the user-creation operation.
4. `BusinessMemory` – Stores the relevant context and operation result.
5. `CriticAgent` – Evaluates the quality and consistency of the final result.

In [149]:
if RUN_COMPONENT_TESTS:
  integrated_create_user_1 = orchestrator_runtime.run(
      "Create a new user named Alice "
      "with phone 0501234567 and balance 1000"
  )

  integrated_create_user_1.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "message": "User created successfully.",
  "user": {
    "user_id": 1,
    "name": "Alice",
    "phone_number": "0501234567"
  },
  "wallet": {
    "user_id": 1,
    "balance": 1000.0
  }
}
Metadata:
{
  "action": "create_user",
  "status": "success",
  "user_id": 1,
  "router_result": {
    "intent": "createUser",
    "confidence": 0.95,
    "explanation": "User creation request with explicit name, phone number, and initial balance.",
    "parameters": {
      "name": "Alice",
      "phone_number": "0501234567",
      "initial_balance": 1000.0
    }
  },
  "orchestrator_decision": {
    "selected_component": "payment_system",
    "action": "create_user",
    "reason": "Router identified intent 'createUser' with high confidence (0.95); route to payment system to create the user.",
    "update_memory": true,
    "run_critic": true
  },
  "critic_result": {
    "quality_score": 5,
    "approved": true,
    "requires

In [150]:
if RUN_COMPONENT_TESTS:
  assert (
      integrated_create_user_1.output["success"]
      is True
  )

  assert (
      integrated_create_user_1.output[
          "user"
      ]["user_id"]
      == 1
  )

  assert (
      integrated_create_user_1.metadata[
          "router_result"
      ]["intent"]
      == Intent.CREATE_USER.value
  )

  assert (
      integrated_create_user_1.metadata[
          "orchestrator_decision"
      ]["selected_component"]
      == "payment_system"
  )

  assert (
      integrated_create_user_1.metadata[
          "critic_result"
      ]["approved"]
      is True
  )

  assert (
      integrated_memory.last_action
      == "create_user"
  )

  print("Integrated create-user test passed.")

Integrated create-user test passed.
time: 996 µs (started: 2026-09-28 13:20:25 +00:00)


### Integration Test: Money Transfer

This test creates a second user and then performs a money transfer through the complete orchestration workflow.

After a successful transfer, the `OrchestratorRuntime` automatically invokes the `FraudDetectionAgent` to assess the transaction, updates the shared `BusinessMemory`, and invokes the `CriticAgent` to evaluate the resulting output.

In [151]:
if RUN_COMPONENT_TESTS:
  integrated_create_user_2 = orchestrator_runtime.run(
      "Create a new user named Bob "
      "with phone 0507654321 and balance 500"
  )

  integrated_transfer = orchestrator_runtime.run(
      "Transfer 100 from user 1 to user 2"
  )

  integrated_transfer.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "message": "Transfer completed successfully.",
  "transaction": {
    "transaction_id": 1,
    "sender_id": 1,
    "receiver_id": 2,
    "amount": 100.0,
    "timestamp": "2026-09-28 13:21:00.072198",
    "status": "completed",
    "risk_score": 0.0
  },
  "sender_balance": 900.0,
  "receiver_balance": 600.0
}
Metadata:
{
  "action": "transfer_money",
  "status": "success",
  "transaction_id": 1,
  "sender_balance_before": 1000.0,
  "router_result": {
    "intent": "transferMoney",
    "confidence": 0.95,
    "explanation": "Explicit instruction to transfer 100 from user 1 to user 2.",
    "parameters": {
      "sender_id": 1,
      "receiver_id": 2,
      "amount": 100.0
    }
  },
  "orchestrator_decision": {
    "selected_component": "payment_system",
    "action": "transfer_money",
    "reason": "Intent 'transferMoney' (confidence 0.95) with sender_id=1, receiver_id=2, amount=100.0 — payment operations must be

In [155]:
if RUN_COMPONENT_TESTS:
  assert integrated_transfer.output["success"] is True

  assert (
      integrated_payment_system.wallets[1].balance
      == 900.0
  )

  assert (
      integrated_payment_system.wallets[2].balance
      == 600.0
  )

  assert (
      integrated_transfer.metadata["fraud_result"]
      is not None
  )

  assert (
      integrated_transfer.metadata["critic_result"]
      is not None
  )

  assert (
      integrated_memory.last_transaction[
          "transaction_id"
      ]
      == 1
  )

  assert (
      integrated_memory.last_action
      == "transfer_money"
  )

  print("Integrated transfer test passed.")

Integrated transfer test passed.
time: 3.27 ms (started: 2026-09-28 13:24:13 +00:00)


### Integration Test: Missing Parameters

This test attempts to perform a money transfer using user names without providing the required numeric user IDs.

The `RouterAgent` may correctly identify the `transferMoney` intent, but the `OrchestratorRuntime` detects that the required sender and receiver IDs are missing and invokes the `FallbackAgent`.

This validates that incomplete requests are handled safely without fabricating missing identifiers or executing an invalid financial operation.

In [156]:
if RUN_COMPONENT_TESTS:
  integrated_missing_fields = (
      orchestrator_runtime.run(
          "Transfer 150 from Alice to Bob"
      )
  )

  integrated_missing_fields.display()

Agent name: FallbackAgent
Confidence: 1.0
Output:
{
  "message": "I cannot complete the transfer because the numeric sender and receiver user IDs are missing. Names like “Alice” and “Bob” are ambiguous; please provide numeric user identifiers.",
  "reason": "Required action parameters are missing: sender_id and receiver_id.",
  "missing_fields": [
    "sender_id",
    "receiver_id"
  ],
  "suggested_request": "Example: \"Transfer 150 from sender_id: <numeric_sender_id> to receiver_id: <numeric_receiver_id>\" (include currency if needed, e.g. 150 USD)."
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "fallback",
  "detected_intent": "transferMoney",
  "router_result": {
    "intent": "transferMoney",
    "confidence": 0.85,
    "explanation": "User asks to transfer 150 but provided names (Alice, Bob) instead of numeric user IDs; sender and receiver IDs are missing.",
    "parameters": {
      "amount": 150.0
    }
  },
  "orchestrator_decision": {
    "selecte

In [157]:
if RUN_COMPONENT_TESTS:
  assert (
      integrated_missing_fields.agent_name
      == "FallbackAgent"
  )

  assert (
      "sender_id"
      in integrated_missing_fields.output[
          "missing_fields"
      ]
  )

  assert (
      "receiver_id"
      in integrated_missing_fields.output[
          "missing_fields"
      ]
  )

  assert (
      len(integrated_payment_system.transactions)
      == 1
  )

  print("Integrated missing-fields test passed.")

Integrated missing-fields test passed.
time: 1.08 ms (started: 2026-09-28 13:24:44 +00:00)


### Integration Test: Last Transaction Explanation

This test asks the system to explain the most recent transaction.

The `RouterAgent` identifies the `explainLastAction` intent, the `OrchestratorAgent` routes the request to the `ExplanationAgent`, and the agent uses the context stored in `BusinessMemory` to generate a structured explanation of the previous transaction.

In [158]:
if RUN_COMPONENT_TESTS:
  integrated_explanation = (
      orchestrator_runtime.run(
          "Explain the last transaction"
      )
  )

  integrated_explanation.display()

Agent name: ExplanationAgent
Confidence: 0.95
Output:
{
  "explanation_type": "action",
  "explanation": "The requested transfer of 150 from Alice to Bob was not completed because the numeric sender_id and receiver_id were missing. Reason: Required action parameters are missing: sender_id and receiver_id. Message: \"I cannot complete the transfer because the numeric sender and receiver user IDs are missing. Names like “Alice” and “Bob” are ambiguous; please provide numeric user identifiers.\" Suggested format: Transfer 150 from sender_id: <numeric_sender_id> to receiver_id: <numeric_receiver_id>.",
  "action": "transferMoney",
  "success": false
}
Metadata:
{
  "provider": "OpenAI",
  "model": "gpt-5-mini",
  "status": "success",
  "last_action": "transferMoney",
  "router_result": {
    "intent": "explainLastAction",
    "confidence": 0.92,
    "explanation": "User asks for an explanation of their most recent transaction; no user_id or transaction identifier was provided.",
    "param

In [161]:
if RUN_COMPONENT_TESTS:
  assert (
      integrated_explanation.agent_name
      == "ExplanationAgent"
  )

  assert (
      integrated_explanation.output[
          "explanation_type"
      ]
      == "action"
  )

  assert (
      "150" in integrated_explanation.output[
          "explanation"
      ]
  )

  print("Integrated explanation test passed.")

Integrated explanation test passed.
time: 3.43 ms (started: 2026-09-28 13:27:03 +00:00)


# JSON Persistence

The system supports saving the current `PaymentSystem` state to a JSON file and restoring it when needed.

The persisted state includes:

- Users.
- Wallets.
- Transactions.
- Payment requests.
- Internal counters used to generate the next unique identifiers.

Persistence is handled entirely through deterministic Python logic and does not require any LLM interaction.

The persistence layer is kept separate from the `PaymentSystem` to maintain a clear separation between business logic and data storage.

In [162]:
class PaymentPersistenceTool:
    """
    Save and restore PaymentSystem data using JSON files.
    """

    def save(
        self,
        payment_system: PaymentSystem,
        file_path: str
    ) -> AgentResult:
        data = {
            "users": [
                asdict(user)
                for user in payment_system.users.values()
            ],
            "wallets": [
                asdict(wallet)
                for wallet in payment_system.wallets.values()
            ],
            "transactions": [
                {
                    **asdict(transaction),
                    "timestamp": transaction.timestamp.isoformat()
                }
                for transaction in payment_system.transactions
            ],
            "payment_requests": [
                {
                    **asdict(payment_request),
                    "created_at": (
                        payment_request.created_at.isoformat()
                    )
                }
                for payment_request
                in payment_system.payment_requests.values()
            ],

            "audit_log": [
                {
                    **log_entry,
                    "timestamp": (
                        log_entry["timestamp"].isoformat()
                        if isinstance(
                            log_entry.get("timestamp"),
                            datetime
                        )
                        else log_entry.get("timestamp")
                    )
                }
                for log_entry in payment_system.audit_log
            ],

            "next_user_id": payment_system.next_user_id,
            "next_transaction_id": (
                payment_system.next_transaction_id
            ),
            "next_request_id": payment_system.next_request_id
        }

        try:
            with open(
                file_path,
                "w",
                encoding="utf-8"
            ) as json_file:
                json.dump(
                    data,
                    json_file,
                    ensure_ascii=False,
                    indent=4
                )

        except OSError as error:
            return AgentResult(
                agent_name="PaymentPersistenceTool",
                output={
                    "success": False,
                    "message": (
                        "Payment-system data could not be saved."
                    ),
                    "error": str(error)
                },
                confidence=1.0,
                metadata={
                    "action": "save_to_json",
                    "status": "rejected",
                    "file_path": file_path
                }
            )

        payment_system._add_audit_log(
            action="save_to_json",
            status="success",
            details={
                "file_path": file_path
            }
        )

        return AgentResult(
            agent_name="PaymentPersistenceTool",
            output={
                "success": True,
                "message": (
                    "Payment-system data saved successfully."
                ),
                "file_path": file_path
            },
            confidence=1.0,
            metadata={
                "action": "save_to_json",
                "status": "success",
                "file_path": file_path
            }
        )

    def load(
        self,
        payment_system: PaymentSystem,
        file_path: str
    ) -> AgentResult:
        try:
            with open(
                file_path,
                "r",
                encoding="utf-8"
            ) as json_file:
                data = json.load(json_file)

        except (OSError, json.JSONDecodeError) as error:
            return AgentResult(
                agent_name="PaymentPersistenceTool",
                output={
                    "success": False,
                    "message": (
                        "Payment-system data could not be loaded."
                    ),
                    "error": str(error)
                },
                confidence=1.0,
                metadata={
                    "action": "load_from_json",
                    "status": "rejected",
                    "file_path": file_path
                }
            )

        payment_system.users = {
            user_data["user_id"]: User(
                user_id=user_data["user_id"],
                name=user_data["name"],
                phone_number=user_data["phone_number"]
            )
            for user_data in data["users"]
        }

        payment_system.wallets = {
            wallet_data["user_id"]: Wallet(
                user_id=wallet_data["user_id"],
                balance=float(wallet_data["balance"])
            )
            for wallet_data in data["wallets"]
        }

        payment_system.transactions = [
            Transaction(
                transaction_id=(
                    transaction_data["transaction_id"]
                ),
                sender_id=transaction_data["sender_id"],
                receiver_id=transaction_data["receiver_id"],
                amount=float(transaction_data["amount"]),
                timestamp=datetime.fromisoformat(
                    transaction_data["timestamp"]
                ),
                status=transaction_data["status"],
                risk_score=float(
                    transaction_data.get(
                        "risk_score",
                        0.0
                    )
                )
            )
            for transaction_data in data["transactions"]
        ]

        payment_system.payment_requests = {
            request_data["request_id"]: PaymentRequest(
                request_id=request_data["request_id"],
                requester_id=request_data["requester_id"],
                payer_id=request_data["payer_id"],
                amount=float(request_data["amount"]),
                status=request_data["status"],
                created_at=datetime.fromisoformat(
                    request_data["created_at"]
                )
            )
            for request_data in data["payment_requests"]
        }

        payment_system.audit_log = [
            {
                **log_entry,
                "timestamp": (
                    datetime.fromisoformat(
                        log_entry["timestamp"]
                    )
                    if log_entry.get("timestamp")
                    else None
                )
            }
            for log_entry in data.get(
                "audit_log",
                []
            )
        ]

        payment_system.log_audit = (
            payment_system.audit_log
        )

        payment_system.next_user_id = data["next_user_id"]
        payment_system.next_transaction_id = (
            data["next_transaction_id"]
        )
        payment_system.next_request_id = data["next_request_id"]

        payment_system._add_audit_log(
            action="load_from_json",
            status="success",
            details={
                "file_path": file_path
            }
        )

        return AgentResult(
            agent_name="PaymentPersistenceTool",
            output={
                "success": True,
                "message": (
                    "Payment-system data loaded successfully."
                ),
                "file_path": file_path
            },
            confidence=1.0,
            metadata={
                "action": "load_from_json",
                "status": "success",
                "file_path": file_path
            }
        )

time: 8.46 ms (started: 2026-09-28 13:27:07 +00:00)


## Persistence Tool Initialization

A `PaymentPersistenceTool` instance is initialized to manage saving and restoring payment-system state.

The tool can persist an existing `PaymentSystem` instance and reconstruct a new instance from the previously saved data.

In [163]:
payment_persistence_tool = PaymentPersistenceTool()

time: 502 µs (started: 2026-09-28 13:27:14 +00:00)


## JSON Persistence Test Setup

A test payment system is initialized with:

- Two users.
- Two corresponding wallets.
- One completed transaction.
- One payment request.

The populated system state is then saved to a JSON file to validate the persistence workflow.

In [164]:
json_test_system = PaymentSystem()

json_test_system.create_user(
    name="Alice",
    phone_number="0501234567",
    initial_balance=800.0
)

json_test_system.create_user(
    name="Bob",
    phone_number="0507654321",
    initial_balance=500.0
)

json_test_system.transfer_money(
    sender_id=1,
    receiver_id=2,
    amount=200.0
)

json_test_system.request_payment(
    requester_id=2,
    payer_id=1,
    amount=50.0
)

print("JSON test system was prepared successfully.")

JSON test system was prepared successfully.
time: 1.89 ms (started: 2026-09-28 13:27:18 +00:00)


## Saving the Payment System

The current state of the payment system is saved to a JSON file named `payment_system_data.json`.

The file stores the system's business entities together with the internal counter values required to preserve identifier generation across saved and restored sessions.

In [165]:
json_file_path = "payment_system_data.json"

save_json_result = payment_persistence_tool.save(
    payment_system=json_test_system,
    file_path=json_file_path
)

save_json_result.display()

Agent name: PaymentPersistenceTool
Confidence: 1.0
Output:
{
  "success": true,
  "message": "Payment-system data saved successfully.",
  "file_path": "payment_system_data.json"
}
Metadata:
{
  "action": "save_to_json",
  "status": "success",
  "file_path": "payment_system_data.json"
}
time: 2.13 ms (started: 2026-09-28 13:27:24 +00:00)


In [166]:
assert save_json_result.output["success"] is True

assert os.path.exists(
    json_file_path
)

assert (
    save_json_result.metadata["action"]
    == "save_to_json"
)

print("Payment-system JSON save test passed.")

Payment-system JSON save test passed.
time: 833 µs (started: 2026-09-28 13:27:28 +00:00)


## JSON File Content

The contents of the saved JSON file are displayed to verify that users, wallets, transactions, and payment requests have been serialized correctly in a readable format.


In [167]:
with open(
    json_file_path,
    "r",
    encoding="utf-8"
) as json_file:
    saved_json_data = json.load(json_file)

print(json.dumps(
    saved_json_data,
    indent=2,
    ensure_ascii=False
))

{
  "users": [
    {
      "user_id": 1,
      "name": "Alice",
      "phone_number": "0501234567"
    },
    {
      "user_id": 2,
      "name": "Bob",
      "phone_number": "0507654321"
    }
  ],
  "wallets": [
    {
      "user_id": 1,
      "balance": 600.0
    },
    {
      "user_id": 2,
      "balance": 700.0
    }
  ],
  "transactions": [
    {
      "transaction_id": 1,
      "sender_id": 1,
      "receiver_id": 2,
      "amount": 200.0,
      "timestamp": "2026-09-28T13:27:18.663506",
      "status": "completed",
      "risk_score": 0.0
    }
  ],
  "payment_requests": [
    {
      "request_id": 1,
      "requester_id": 2,
      "payer_id": 1,
      "amount": 50.0,
      "status": "pending",
      "created_at": "2026-09-28T13:27:18.663655"
    }
  ],
  "audit_log": [
    {
      "timestamp": "2026-09-28T13:27:18.663130",
      "action": "create_user",
      "status": "success",
      "details": {
        "user_id": 1,
        "name": "Alice",
        "phone_number": "050123

## Loading Data into a New Payment System

A new, empty `PaymentSystem` instance is created and populated with the data restored from the saved JSON file.

This verifies that the complete business state can be reconstructed in a new system instance while preserving the previously stored data.


In [168]:
loaded_payment_system = PaymentSystem()

load_json_result = payment_persistence_tool.load(
    payment_system=loaded_payment_system,
    file_path=json_file_path
)

load_json_result.display()

Agent name: PaymentPersistenceTool
Confidence: 1.0
Output:
{
  "success": true,
  "message": "Payment-system data loaded successfully.",
  "file_path": "payment_system_data.json"
}
Metadata:
{
  "action": "load_from_json",
  "status": "success",
  "file_path": "payment_system_data.json"
}
time: 2.27 ms (started: 2026-09-28 13:27:47 +00:00)


### User and Wallet Loading Validation

The validation confirms that both users and their corresponding wallets are restored correctly, including the previously saved user data and wallet balances.

In [169]:
assert load_json_result.output["success"] is True

assert len(loaded_payment_system.users) == 2
assert len(loaded_payment_system.wallets) == 2

assert loaded_payment_system.users[1].name == "Alice"
assert loaded_payment_system.users[2].name == "Bob"

assert (
    loaded_payment_system.users[1].phone_number
    == "0501234567"
)

assert (
    loaded_payment_system.users[2].phone_number
    == "0507654321"
)

assert loaded_payment_system.wallets[1].balance == 600.0
assert loaded_payment_system.wallets[2].balance == 700.0

print("Loaded users and wallets were validated.")

Loaded users and wallets were validated.
time: 1.28 ms (started: 2026-09-28 13:27:51 +00:00)


### Transaction Loading Validation

The validation confirms that the transaction is successfully restored with its original identifiers, amount, status, and timestamp.

In [170]:
assert len(
    loaded_payment_system.transactions
) == 1

loaded_transaction = (
    loaded_payment_system.transactions[0]
)

assert loaded_transaction.transaction_id == 1
assert loaded_transaction.sender_id == 1
assert loaded_transaction.receiver_id == 2
assert loaded_transaction.amount == 200.0
assert loaded_transaction.status == "completed"

assert isinstance(
    loaded_transaction.timestamp,
    datetime
)

print("Loaded transactions were validated.")

Loaded transactions were validated.
time: 2.36 ms (started: 2026-09-28 13:27:54 +00:00)


### Payment Request Loading Validation

The validation confirms that the payment request is successfully restored with its original identifiers, amount, status, and creation timestamp.

In [171]:
assert len(
    loaded_payment_system.payment_requests
) == 1

loaded_payment_request = (
    loaded_payment_system.payment_requests[1]
)

assert loaded_payment_request.request_id == 1
assert loaded_payment_request.requester_id == 2
assert loaded_payment_request.payer_id == 1
assert loaded_payment_request.amount == 50.0
assert loaded_payment_request.status == "pending"

assert isinstance(
    loaded_payment_request.created_at,
    datetime
)

print("Loaded payment requests were validated.")

Loaded payment requests were validated.
time: 1.21 ms (started: 2026-09-28 13:27:58 +00:00)


### Future Identifier Validation

The validation confirms that the system's internal identifier counters are restored correctly.

Preserving these counters ensures that newly created users, transactions, and payment requests receive unique identifiers after the system state has been restored, preventing duplicate IDs.

In [172]:
assert loaded_payment_system.next_user_id == 3
assert loaded_payment_system.next_transaction_id == 2
assert loaded_payment_system.next_request_id == 2

print("Loaded identifier counters were validated.")

Loaded identifier counters were validated.
time: 792 µs (started: 2026-09-28 13:28:13 +00:00)


## Continuing Operations After System Restoration

After restoring the saved system state, a new user is created to verify that normal operations can continue correctly.

The new user is expected to receive ID `3`, confirming that the internal identifier counters were successfully restored from the JSON file.

In [173]:
loaded_new_user_result = (
    loaded_payment_system.create_user(
        name="Charlie",
        phone_number="0509999999",
        initial_balance=300.0
    )
)

loaded_new_user_result.display()

Agent name: PaymentSystem
Confidence: 1.0
Output:
{
  "success": true,
  "message": "User created successfully.",
  "user": {
    "user_id": 3,
    "name": "Charlie",
    "phone_number": "0509999999"
  },
  "wallet": {
    "user_id": 3,
    "balance": 300.0
  }
}
Metadata:
{
  "action": "create_user",
  "status": "success",
  "user_id": 3
}
time: 779 µs (started: 2026-09-28 13:28:16 +00:00)


### System Continuity Validation

The validation confirms that the restored system continues to operate correctly and that newly created users receive unique identifiers without duplicating existing user IDs.

In [174]:
assert (
    loaded_new_user_result.output["user"]["user_id"]
    == 3
)

assert len(
    loaded_payment_system.users
) == 3

assert loaded_payment_system.next_user_id == 4

print("Payment-system continuity test passed.")

Payment-system continuity test passed.
time: 737 µs (started: 2026-09-28 13:28:21 +00:00)


## Missing File Handling Test

This test attempts to load payment-system data from a file that does not exist.

The persistence tool is expected to handle the missing file gracefully by returning a structured error result rather than interrupting execution.

In [175]:
missing_json_file_result = (
    payment_persistence_tool.load(
        payment_system=PaymentSystem(),
        file_path="missing_payment_system_file.json"
    )
)

missing_json_file_result.display()

Agent name: PaymentPersistenceTool
Confidence: 1.0
Output:
{
  "success": false,
  "message": "Payment-system data could not be loaded.",
  "error": "[Errno 2] No such file or directory: 'missing_payment_system_file.json'"
}
Metadata:
{
  "action": "load_from_json",
  "status": "rejected",
  "file_path": "missing_payment_system_file.json"
}
time: 974 µs (started: 2026-09-28 13:28:24 +00:00)


## Missing File Handling Test

This test attempts to load payment-system data from a file that does not exist.

The persistence tool is expected to handle the missing file gracefully by returning a structured error result rather than interrupting execution.

In [176]:
assert (
    missing_json_file_result.output["success"]
    is False
)

assert (
    missing_json_file_result.metadata["status"]
    == "rejected"
)

assert (
    missing_json_file_result.metadata["action"]
    == "load_from_json"
)

print("Missing JSON file test passed.")

Missing JSON file test passed.
time: 785 µs (started: 2026-09-28 13:28:28 +00:00)


# Final Integration Tests

This section validates complete end-to-end workflows through the `OrchestratorRuntime`.

Each request is processed through the `RouterAgent`, `OrchestratorAgent`, the appropriate deterministic tools or specialized agents, `BusinessMemory`, and the `CriticAgent`, with additional evaluation agents invoked when required by the workflow.

A fresh `PaymentSystem` and `BusinessMemory` instance are initialized for these tests to ensure that each scenario is evaluated in a controlled environment without depending on state created by previous notebook cells.

In [177]:
final_payment_system = PaymentSystem()
final_business_memory = BusinessMemory()

final_runtime = OrchestratorRuntime(
    router_agent=router_agent,
    orchestrator_agent=orchestrator_agent,
    payment_system=final_payment_system,
    memory=final_business_memory,
    fallback_agent=fallback_agent,
    fraud_detection_agent=fraud_detection_agent,
    security_agent=security_agent,
    explanation_agent=explanation_agent,
    critic_agent=critic_agent,
    reflection_agent=reflection_agent
)

final_test_results: Dict[str, AgentResult] = {}

print("Final integration-test environment was created.")

Final integration-test environment was created.
time: 1.65 ms (started: 2026-09-28 13:28:31 +00:00)


## Test 1: Creating Two Users

This test creates two users through natural-language requests processed by the complete orchestration workflow.

The `PaymentSystem` generates a unique identifier for each user and creates a corresponding wallet with the specified initial balance.


In [178]:
final_test_results["create_user_1"] = final_runtime.run(
    "Create a new user named Alice "
    "with phone 0501111111 and balance 2000"
)

display_agent_execution(
    "TEST 1 - CREATE USER 1",
    "Create a new user named Alice with phone 0501111111 and balance 2000",
    final_test_results["create_user_1"]
)

final_test_results["create_user_2"] = final_runtime.run(
    "Create a new user named Bob "
    "with phone 0502222222 and balance 500"
)

display_agent_execution(
    "TEST 1 - CREATE USER 2",
    "Create a new user named Bob with phone 0502222222 and balance 500",
    final_test_results["create_user_2"]
)

assert (
    final_test_results["create_user_1"]
    .output["success"]
    is True
)

assert (
    final_test_results["create_user_2"]
    .output["success"]
    is True
)

assert len(final_payment_system.users) == 2
assert len(final_payment_system.wallets) == 2

assert final_payment_system.users[1].name == "Alice"
assert final_payment_system.users[2].name == "Bob"

assert final_payment_system.wallets[1].balance == 2000.0
assert final_payment_system.wallets[2].balance == 500.0

print("PASS 1: Two users were created through OrchestratorRuntime.")


TEST 1 - CREATE USER 1

 USER
----------------------------------------------------------------------------------------------------
Create a new user named Alice with phone 0501111111 and balance 2000

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "createUser",
  "confidence": 0.98,
  "explanation": "User creation request specifying name, phone number, and initial balance.",
  "parameters": {
    "name": "Alice",
    "phone_number": "0501111111",
    "initial_balance": 2000.0
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "create_user",
  "reason": "Intent 'createUser' maps to payment_system for creating a new user with provided name, phone_number, and initial_balance.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
-----------------------------------

## Test 2: Balance Verification

This test retrieves the balances of both users through the complete integrated workflow.

The `RouterAgent` is expected to identify the `checkBalance` intent, while the `OrchestratorAgent` routes the request to the `PaymentSystem` for execution.

In [179]:
final_test_results["balance_1"] = final_runtime.run(
    "Check the balance of user 1"
)

display_agent_execution(
    "TEST 2A - CHECK BALANCE USER 1",
    "Check the balance of user 1",
    final_test_results["balance_1"]
)


final_test_results["balance_2"] = final_runtime.run(
    "Check the balance of user 2"
)

display_agent_execution(
    "TEST 2B - CHECK BALANCE USER 2",
    "Check the balance of user 2",
    final_test_results["balance_2"]
)


assert (
    final_test_results["balance_1"]
    .output["balance"]
    == 2000.0
)

assert (
    final_test_results["balance_2"]
    .output["balance"]
    == 500.0
)

print("PASS 2: User balances were retrieved correctly.")


TEST 2A - CHECK BALANCE USER 1

 USER
----------------------------------------------------------------------------------------------------
Check the balance of user 1

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "checkBalance",
  "confidence": 0.95,
  "explanation": "Request to check balance for user id 1.",
  "parameters": {
    "user_id": 1
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "get_balance",
  "reason": "Router identified intent 'checkBalance' with parameter user_id=1; route to payment_system to retrieve balance.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
----------------------------------------------------------------------------------------------------
Not Executed

SecurityAgent
--------------------------------------------------

## Test 3: Successful Money Transfer

This test performs a transfer of 100 from User 1 to User 2 through the complete orchestration workflow.

After the transfer is completed successfully, the `FraudDetectionAgent` is automatically invoked to evaluate the transaction for potentially suspicious activity.

In [180]:
user_message = (
    "Transfer 100 from user 1 to user 2"
)


final_test_results["valid_transfer"] = final_runtime.run(
    user_message
)

display_agent_execution(
    "TEST 3 - VALID TRANSFER",
    user_message,
    final_test_results["valid_transfer"]
)

valid_transfer_result = final_test_results["valid_transfer"]

assert valid_transfer_result.output["success"] is True

assert final_payment_system.wallets[1].balance == 1900.0
assert final_payment_system.wallets[2].balance == 600.0

assert len(final_payment_system.transactions) == 1

assert (
    valid_transfer_result.metadata["fraud_result"]
    is not None
)

assert (
    valid_transfer_result.metadata["critic_result"]
    is not None
)

print("PASS 3: Valid transfer completed and fraud review was activated.")


TEST 3 - VALID TRANSFER

 USER
----------------------------------------------------------------------------------------------------
Transfer 100 from user 1 to user 2

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "transferMoney",
  "confidence": 0.98,
  "explanation": "Explicit instruction to transfer 100 from user 1 to user 2.",
  "parameters": {
    "sender_id": 1,
    "receiver_id": 2,
    "amount": 100.0
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "transfer_money",
  "reason": "Router identified intent 'transferMoney' (transfer 100.0 from sender_id 1 to receiver_id 2). Payment operations must be handled by the payment_system.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
----------------------------------------------------------------------

## Test 4: Negative Transfer Amount

This test attempts to perform a money transfer using a negative amount.

The `PaymentSystem` is expected to reject the invalid operation, after which the `ReflectionAgent` provides structured feedback explaining the failure and suggesting a valid correction.

In [181]:
user_message = (
    "Transfer -50 from user 1 to user 2"
)

final_test_results["negative_transfer"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 4 - NEGATIVE TRANSFER",
    user_message,
    final_test_results["negative_transfer"]
)

negative_transfer_result = (
    final_test_results["negative_transfer"]
)

assert (
    negative_transfer_result.output["success"]
    is False
)

assert (
    "greater than zero"
    in negative_transfer_result.output["message"].lower()
)

assert (
    negative_transfer_result.metadata[
        "reflection_result"
    ]
    is not None
)

assert final_payment_system.wallets[1].balance == 1900.0
assert final_payment_system.wallets[2].balance == 600.0

print("PASS 4: Negative transfer was rejected.")


TEST 4 - NEGATIVE TRANSFER

 USER
----------------------------------------------------------------------------------------------------
Transfer -50 from user 1 to user 2

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "transferMoney",
  "confidence": 0.87,
  "explanation": "User requests a money transfer from user 1 to user 2; amount given as -50 (negative).",
  "parameters": {
    "sender_id": 1,
    "receiver_id": 2,
    "amount": -50.0
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "transfer_money",
  "reason": "Intent 'transferMoney' maps to payment_system. Router parameters include sender_id=1, receiver_id=2, amount=-50.0 (negative) — payment system should validate/handle invalid amount.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
-----------

## Test 5: Insufficient Funds

This test attempts to transfer an amount that exceeds the sender's available balance.

The `PaymentSystem` is expected to reject the operation without modifying either user's wallet balance.

In [182]:
user_message = (
    "Transfer 5000 from user 2 to user 1"
)

final_test_results["insufficient_balance"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 5 - INSUFFICIENT BALANCE",
    user_message,
    final_test_results["insufficient_balance"]
)

insufficient_result = (
    final_test_results["insufficient_balance"]
)

assert insufficient_result.output["success"] is False

assert (
    insufficient_result.output["message"]
    == "Insufficient balance."
)

assert (
    insufficient_result.metadata["reflection_result"]
    is not None
)

assert final_payment_system.wallets[1].balance == 1900.0
assert final_payment_system.wallets[2].balance == 600.0

print("PASS 5: Insufficient-balance transfer was rejected.")


TEST 5 - INSUFFICIENT BALANCE

 USER
----------------------------------------------------------------------------------------------------
Transfer 5000 from user 2 to user 1

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "transferMoney",
  "confidence": 0.95,
  "explanation": "User explicitly requests transferring 5000 from user 2 (sender) to user 1 (receiver).",
  "parameters": {
    "sender_id": 2,
    "receiver_id": 1,
    "amount": 5000.0
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "transfer_money",
  "reason": "Intent is transferMoney with high confidence (0.95); user requested transferring 5000 from sender_id 2 to receiver_id 1. Payment operations must be handled by payment_system.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
------------

## Test 6: Non-Existent User

This test attempts to transfer money to a user ID that does not exist in the system.

The `PaymentSystem` is expected to reject the operation and record the failed attempt in the `audit_log`.


In [183]:
user_message = (
    "Transfer 50 from user 1 to user 999"
)

final_test_results["missing_user"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 6 - TRANSFER TO MISSING USER",
    user_message,
    final_test_results["missing_user"]
)

missing_user_result = (
    final_test_results["missing_user"]
)

assert missing_user_result.output["success"] is False

assert (
    missing_user_result.output["message"]
    == "Receiver does not exist."
)

assert len(final_payment_system.transactions) == 1

print("PASS 6: Transfer to a missing user was rejected.")


TEST 6 - TRANSFER TO MISSING USER

 USER
----------------------------------------------------------------------------------------------------
Transfer 50 from user 1 to user 999

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "transferMoney",
  "confidence": 0.95,
  "explanation": "User asked to transfer 50 from user 1 to user 999.",
  "parameters": {
    "sender_id": 1,
    "receiver_id": 999,
    "amount": 50.0
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "transfer_money",
  "reason": "Router identified intent transferMoney: user requested transfer of 50.0 from sender_id 1 to receiver_id 999.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
----------------------------------------------------------------------------------------------------
Not Exec

## Test 7: Self-Transfer Prevention

This test attempts to perform a money transfer where the sender and receiver are the same user.

The `PaymentSystem` is expected to reject the operation, preventing a user from transferring funds to their own account.

In [184]:
user_message = (
    "Transfer 50 from user 1 to user 1"
)

final_test_results["self_transfer"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 7 - SELF TRANSFER",
    user_message,
    final_test_results["self_transfer"]
)

self_transfer_result = (
    final_test_results["self_transfer"]
)

assert self_transfer_result.output["success"] is False

assert (
    "themselves"
    in self_transfer_result.output["message"].lower()
)

assert len(final_payment_system.transactions) == 1

print("PASS 7: Self-transfer was rejected.")



TEST 7 - SELF TRANSFER

 USER
----------------------------------------------------------------------------------------------------
Transfer 50 from user 1 to user 1

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "transferMoney",
  "confidence": 0.9,
  "explanation": "User requested a transfer of 50 from user 1 to user 1.",
  "parameters": {
    "sender_id": 1,
    "receiver_id": 1,
    "amount": 50.0
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "transfer_money",
  "reason": "Intent 'transferMoney' (confidence 0.9) with parameters sender_id=1, receiver_id=1, amount=50.0 — route to payment_system to perform transfer_money.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
---------------------------------------------------------------------------------

## Test 8: Payment Request Creation and Approval

This test creates a payment request in which User 2 requests 100 from User 1.

The request is then approved, which is expected to create a new transaction and transfer 100 from User 1 to User 2.

In [185]:
user_message = (
    "User 2 requests 100 from user 1"
)

final_test_results["payment_request"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 8A - CREATE PAYMENT REQUEST",
    user_message,
    final_test_results["payment_request"]
)

payment_request_result = (
    final_test_results["payment_request"]
)

assert payment_request_result.output["success"] is True

assert (
    final_payment_system.payment_requests[1].status
    == "pending"
)


user_message = (
    "Approve payment request 1"
)

final_test_results["approve_request"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 8B - APPROVE PAYMENT REQUEST",
    user_message,
    final_test_results["approve_request"]
)

approve_request_result = (
    final_test_results["approve_request"]
)

assert approve_request_result.output["success"] is True

assert (
    final_payment_system.payment_requests[1].status
    == "approved"
)

assert final_payment_system.wallets[1].balance == 1800.0
assert final_payment_system.wallets[2].balance == 700.0

assert len(final_payment_system.transactions) == 2

print("PASS 8: Payment request was created and approved.")


TEST 8A - CREATE PAYMENT REQUEST

 USER
----------------------------------------------------------------------------------------------------
User 2 requests 100 from user 1

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "requestPayment",
  "confidence": 0.95,
  "explanation": "User 2 is requesting a payment of 100 from user 1.",
  "parameters": {
    "requester_id": 2,
    "payer_id": 1,
    "amount": 100.0
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "request_payment",
  "reason": "Router identified intent 'requestPayment' (confidence 0.95) requesting 100.0 from payer_id 1 by requester_id 2; payment operations must be handled by payment_system.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
--------------------------------------------------------

## Test 9: Duplicate Payment Approval

This test attempts to approve a payment request that has already been approved.

The `PaymentSystem` is expected to reject the duplicate approval and ensure that no additional transaction is created.

In [186]:
user_message = (
    "Approve payment request 1"
)

final_test_results["double_approval"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 9 - DUPLICATE APPROVAL",
    user_message,
    final_test_results["double_approval"]
)

double_approval_result = (
    final_test_results["double_approval"]
)

assert double_approval_result.output["success"] is False

assert (
    final_payment_system.payment_requests[1].status
    == "approved"
)

assert len(final_payment_system.transactions) == 2

assert final_payment_system.wallets[1].balance == 1800.0
assert final_payment_system.wallets[2].balance == 700.0

print("PASS 9: Duplicate approval was rejected.")


TEST 9 - DUPLICATE APPROVAL

 USER
----------------------------------------------------------------------------------------------------
Approve payment request 1

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "approvePayment",
  "confidence": 0.98,
  "explanation": "User explicitly asked to approve payment request #1.",
  "parameters": {
    "request_id": 1
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "approve_payment_request",
  "reason": "Intent is approvePayment with high confidence and request_id present; route to payment system to approve the payment request.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
----------------------------------------------------------------------------------------------------
Not Executed

SecurityAgent
----------

## Test 10: Payment Request Rejection

This test creates a new payment request and then rejects it.

The rejection is expected to update the request status without creating a transaction or modifying either user's wallet balance.

In [187]:
user_message = (
    "User 1 requests 50 from user 2"
)

final_test_results["second_payment_request"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 10A - CREATE PAYMENT REQUEST",
    user_message,
    final_test_results["second_payment_request"]
)

assert (
    final_test_results["second_payment_request"]
    .output["success"]
    is True
)


user_message = (
    "Reject payment request 2"
)

final_test_results["reject_request"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 10B - REJECT PAYMENT REQUEST",
    user_message,
    final_test_results["reject_request"]
)

reject_request_result = (
    final_test_results["reject_request"]
)

assert reject_request_result.output["success"] is True

assert (
    final_payment_system.payment_requests[2].status
    == "rejected"
)

assert len(final_payment_system.transactions) == 2

assert final_payment_system.wallets[1].balance == 1800.0
assert final_payment_system.wallets[2].balance == 700.0

print("PASS 10: Payment request was rejected without a transfer.")


TEST 10A - CREATE PAYMENT REQUEST

 USER
----------------------------------------------------------------------------------------------------
User 1 requests 50 from user 2

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "requestPayment",
  "confidence": 0.96,
  "explanation": "User 1 is requesting a payment of 50 from user 2.",
  "parameters": {
    "requester_id": 1,
    "payer_id": 2,
    "amount": 50.0
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "request_payment",
  "reason": "Router identified intent 'requestPayment' with high confidence (0.96): requester_id 1 requests 50.0 from payer_id 2.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
----------------------------------------------------------------------------------------------------
Not Ex

## Test 11: Suspicious Transaction Detection

This test performs a transfer of 1500 from a sender with an available balance of 1800.

The transaction exceeds the configured amount threshold, represents more than 80% of the sender's available balance, and occurs after multiple previous transactions.

Under these conditions, the `FraudDetectionAgent` is expected to classify the transaction as suspicious, after which the `SecurityAgent` is automatically invoked to perform an additional security review.

In [188]:
user_message = (
    "Transfer 1500 from user 1 to user 2"
)

final_test_results["suspicious_transfer"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 11 - SUSPICIOUS TRANSACTION",
    user_message,
    final_test_results["suspicious_transfer"]
)

suspicious_transfer_result = (
    final_test_results["suspicious_transfer"]
)

assert suspicious_transfer_result.output["success"] is True

fraud_metadata = (
    suspicious_transfer_result
    .metadata["fraud_result"]
)

assert fraud_metadata is not None
assert fraud_metadata["suspicious"] is True
assert fraud_metadata["risk_score"] >= 0.70

assert (
    suspicious_transfer_result
    .metadata["security_result"]
    is not None
)

assert final_payment_system.wallets[1].balance == 300.0
assert final_payment_system.wallets[2].balance == 2200.0

assert len(final_payment_system.transactions) == 3

print(
    "PASS 11: Suspicious transaction activated fraud and security agents."
)


TEST 11 - SUSPICIOUS TRANSACTION

 USER
----------------------------------------------------------------------------------------------------
Transfer 1500 from user 1 to user 2

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "transferMoney",
  "confidence": 0.95,
  "explanation": "User requests a money transfer specifying sender user 1, receiver user 2, amount 1500.",
  "parameters": {
    "sender_id": 1,
    "receiver_id": 2,
    "amount": 1500.0
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "transfer_money",
  "reason": "Router identified intent 'transferMoney' (confidence 0.95) with parameters sender_id=1, receiver_id=2, amount=1500.0; route to payment system for executing the transfer.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
-------------

## Test 12: Last Transaction Explanation

This test asks the system to explain the most recent transaction.

The `ExplanationAgent` is expected to use the context stored in `BusinessMemory` and generate an explanation referring to the previous transfer of 1500.

In [189]:
user_message = (
    "Explain the last transaction"
)

final_test_results["explain_last"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 12 - EXPLAIN LAST TRANSACTION",
    user_message,
    final_test_results["explain_last"]
)

explain_last_result = (
    final_test_results["explain_last"]
)

assert (
    explain_last_result.agent_name
    == "ExplanationAgent"
)

assert (
    explain_last_result.output["explanation_type"]
    == "transaction"
)

assert (
    "1500"
    in explain_last_result.output["explanation"]
)

assert (
    final_business_memory.last_transaction
    is not None
)

print(
    "PASS 12: Last transaction was explained from business memory."
)


TEST 12 - EXPLAIN LAST TRANSACTION

 USER
----------------------------------------------------------------------------------------------------
Explain the last transaction

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "explainLastAction",
  "confidence": 0.92,
  "explanation": "User requests an explanation of their most recent transaction; no user or transaction identifiers were provided.",
  "parameters": {}
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "explanation_agent",
  "action": "explain_last_action",
  "reason": "Intent is explainLastAction. The user asked for an explanation of their most recent transaction; explanation_agent is responsible for providing explanations. No identifiers provided so agent should use the user's last transaction from memory.",
  "update_memory": true,
  "run_critic": t

## Test 13: Transaction History

This test requests the transaction history for User 1.

The system is expected to return all three transactions in which User 1 participated.

In [190]:
user_message = (
    "Show transactions for user 1"
)

final_test_results["show_transactions"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 13 - SHOW TRANSACTION HISTORY",
    user_message,
    final_test_results["show_transactions"]
)

transactions_result = (
    final_test_results["show_transactions"]
)

assert transactions_result.output["success"] is True

assert (
    len(transactions_result.output["transactions"])
    == 3
)

print(
    "PASS 13: Transaction history was returned correctly."
)


TEST 13 - SHOW TRANSACTION HISTORY

 USER
----------------------------------------------------------------------------------------------------
Show transactions for user 1

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "showTransactions",
  "confidence": 0.95,
  "explanation": "User asked to show transactions for user ID 1.",
  "parameters": {
    "user_id": 1
  }
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "payment_system",
  "action": "get_transactions",
  "reason": "Router intent 'showTransactions' maps to payment_system action get_transactions; router provided user_id parameter.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
----------------------------------------------------------------------------------------------------
Not Executed

SecurityAgent
------------------------

## Test 14: Security Review

This test initiates an explicit security review through the integrated system.

The `RouterAgent` is expected to identify the `securityReview` intent, while the `OrchestratorAgent` routes the request to the `SecurityAgent` for evaluation.

In [192]:
user_message = (
    "Run a security review"
)

final_test_results["security_review"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 14 - SECURITY REVIEW",
    user_message,
    final_test_results["security_review"]
)

security_review_result = (
    final_test_results["security_review"]
)

assert (
    security_review_result.agent_name
    == "SecurityAgent"
)

assert security_review_result.output["passed"] is True

print("PASS 14: Security review passed.")


TEST 14 - SECURITY REVIEW

 USER
----------------------------------------------------------------------------------------------------
Run a security review

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "securityReview",
  "confidence": 0.92,
  "explanation": "User explicitly requested a security review; no business parameters were provided.",
  "parameters": {}
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "security_agent",
  "action": "security_review",
  "reason": "Router identified intent 'securityReview' with high confidence; user requested a security review and no parameters were provided.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
----------------------------------------------------------------------------------------------------
Not Executed

SecurityAgent
-------------

## Test 15: On-Demand Fraud Analysis

This test explicitly requests a fraud assessment of the most recent transaction.

The `FraudDetectionAgent` is expected to retrieve the transaction context stored in `BusinessMemory` and use it to perform a structured fraud-risk assessment.

In [191]:
user_message = (
    "Check the last transaction for fraud"
)

final_test_results["fraud_check"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 15 - FRAUD CHECK",
    user_message,
    final_test_results["fraud_check"]
)

fraud_check_result = (
    final_test_results["fraud_check"]
)

assert (
    fraud_check_result.agent_name
    == "FraudDetectionAgent"
)

assert "risk_score" in fraud_check_result.output
assert "suspicious" in fraud_check_result.output

print("PASS 15: Explicit fraud check was completed.")


TEST 15 - FRAUD CHECK

 USER
----------------------------------------------------------------------------------------------------
Check the last transaction for fraud

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "fraudCheck",
  "confidence": 0.92,
  "explanation": "User asks to check the last transaction for fraud; no user_id or transaction identifier provided.",
  "parameters": {}
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
{
  "selected_component": "fraud_detection_agent",
  "action": "fraud_check",
  "reason": "Intent 'fraudCheck' detected. No transaction identifier was provided in parameters; memory indicates a last transaction is available, so route to fraud_detection_agent to perform fraud_check using the last-transaction context.",
  "update_memory": true,
  "run_critic": true
}

FraudDetectionAgent
--------------------

## Test 16: Unrecognized Request

This test submits a request that does not correspond to any supported business operation.

The `RouterAgent` is expected to classify the intent as `unknown`, after which the `FallbackAgent` is invoked to provide appropriate guidance to the user.

In [193]:
user_message = "?"

final_test_results["unknown_request"] = (
    final_runtime.run(user_message)
)

display_agent_execution(
    "TEST 16 - UNKNOWN REQUEST",
    user_message,
    final_test_results["unknown_request"]
)

unknown_result = (
    final_test_results["unknown_request"]
)

assert (
    unknown_result.agent_name
    == "FallbackAgent"
)

assert (
    unknown_result.metadata["status"]
    == "fallback"
)

assert (
    final_business_memory.last_intent
    == Intent.UNKNOWN.value
)

print(
    "PASS 16: Unclear request activated FallbackAgent."
)


TEST 16 - UNKNOWN REQUEST

 USER
----------------------------------------------------------------------------------------------------
?

RouterAgent
----------------------------------------------------------------------------------------------------
{
  "intent": "unknown",
  "confidence": 0.2,
  "explanation": "Single-question mark; user intent is not identifiable.",
  "parameters": {}
}

OrchestratorAgent
----------------------------------------------------------------------------------------------------
Not Executed

FraudDetectionAgent
----------------------------------------------------------------------------------------------------
Not Executed

SecurityAgent
----------------------------------------------------------------------------------------------------
Not Executed

ReflectionAgent
----------------------------------------------------------------------------------------------------
Not Executed

CriticAgent
------------------------------------------------------------------

# Integration Test Summary

This section summarizes the integration-test scenarios and identifies the component responsible for producing each final result.

All scenarios were executed through the `OrchestratorRuntime`, validating the complete coordinated workflow rather than invoking the `PaymentSystem` directly.

In [194]:
print("\n")
print("=" * 110)
print("FINAL INTEGRATION TEST SUMMARY")
print("=" * 110)

print(
    f"{'TEST':<30}"
    f"{'FINAL AGENT':<25}"
    f"{'ROUTER':<10}"
    f"{'ORCHESTRATOR':<15}"
    f"{'FRAUD':<10}"
    f"{'SECURITY':<10}"
    f"{'REFLECTION':<12}"
    f"{'CRITIC':<10}"
)

print("-" * 110)

for test_name, test_result in final_test_results.items():

    metadata = test_result.metadata or {}

    print(
        f"{test_name:<30}"
        f"{test_result.agent_name:<25}"
        f"{'✓' if metadata.get('router_result') else '-':<10}"
        f"{'✓' if metadata.get('orchestrator_decision') else '-':<15}"
        f"{'✓' if metadata.get('fraud_result') else '-':<10}"
        f"{'✓' if metadata.get('security_result') else '-':<10}"
        f"{'✓' if metadata.get('reflection_result') else '-':<12}"
        f"{'✓' if metadata.get('critic_result') else '-':<10}"
    )

print("=" * 110)
print(f"Total completed scenarios : {len(final_test_results)}")
print("=" * 110)



FINAL INTEGRATION TEST SUMMARY
TEST                          FINAL AGENT              ROUTER    ORCHESTRATOR   FRAUD     SECURITY  REFLECTION  CRITIC    
--------------------------------------------------------------------------------------------------------------
create_user_1                 PaymentSystem            ✓         ✓              -         -         -           ✓         
create_user_2                 PaymentSystem            ✓         ✓              -         -         -           ✓         
balance_1                     PaymentSystem            ✓         ✓              -         -         -           ✓         
balance_2                     PaymentSystem            ✓         ✓              -         -         -           ✓         
valid_transfer                PaymentSystem            ✓         ✓              ✓         -         -           ✓         
negative_transfer             PaymentSystem            ✓         ✓              -         -         ✓           ✓     

## Final System State Validation

After completing all integration scenarios, the final system state is validated to ensure that it accurately reflects the operations performed throughout the workflow.

The expected final state includes:

- Two registered users.
- Three completed transactions.
- Two payment requests.
- One approved payment request.
- One rejected payment request.
- A non-empty `audit_log`.
- An active `BusinessMemory` containing the latest workflow context.

In [195]:
assert len(final_payment_system.users) == 2
assert len(final_payment_system.wallets) == 2

assert len(final_payment_system.transactions) == 3
assert len(final_payment_system.payment_requests) == 2

assert (
    final_payment_system.payment_requests[1].status
    == "approved"
)

assert (
    final_payment_system.payment_requests[2].status
    == "rejected"
)

assert len(final_payment_system.audit_log) > 0

assert (
    len(final_business_memory.recent_actions)
    <= 10
)


print("\nFINAL SYSTEM STATE")
print("-" * 80)

print("Users:", len(final_payment_system.users))
print("Wallets:", len(final_payment_system.wallets))
print("Transactions:", len(final_payment_system.transactions))
print(
    "Payment Requests:",
    len(final_payment_system.payment_requests)
)
print(
    "Audit Log Entries:",
    len(final_payment_system.audit_log)
)
print(
    "Recent Memory Actions:",
    len(final_business_memory.recent_actions)
)


print("\nBUSINESS MEMORY")
print("-" * 80)

last_intent = final_business_memory.last_intent

if isinstance(last_intent, Intent):
    last_intent = last_intent.value

print("Last Recorded Intent:")
print(last_intent)

print("\nLast Business User:")
pprint(final_business_memory.last_user)

print("\nLast Successful Transaction:")
pprint(final_business_memory.last_transaction)

print("\nLast Payment Request:")
pprint(final_business_memory.last_payment_request)

print("\nLast Recorded Result:")
pprint(final_business_memory.last_result)

print("\nRecent Business Actions:")

for index, action in enumerate(
    final_business_memory.recent_actions,
    start=1
):
    if isinstance(action, Intent):
        action = action.value

    print(f"{index:>2}. {action}")


print("\nFinal system-state validation passed.")
print("All mandatory integration scenarios passed.")


FINAL SYSTEM STATE
--------------------------------------------------------------------------------
Users: 2
Wallets: 2
Transactions: 3
Payment Requests: 2
Audit Log Entries: 17
Recent Memory Actions: 10

BUSINESS MEMORY
--------------------------------------------------------------------------------
Last Recorded Intent:
unknown

Last Business User:
{'name': 'Alice', 'phone_number': '0501111111', 'user_id': 1}

Last Successful Transaction:
{'amount': 1500.0,
 'receiver_id': 2,
 'risk_score': 1.0,
 'sender_id': 1,
 'status': 'completed',
 'timestamp': datetime.datetime(2026, 9, 28, 13, 37, 8, 96605),
 'transaction_id': 3}

Last Payment Request:
{'amount': 50.0,
 'created_at': datetime.datetime(2026, 9, 28, 13, 36, 25, 249656),
 'payer_id': 2,
 'request_id': 2,
 'requester_id': 1,
 'status': 'rejected'}

Last Recorded Result:
{'message': 'I can’t act on this because your message is unclear. I need a '
            'specific action and details to process a payment-related request.',
 'mi